# Portfolio edition
Nicholas Koch | Sports analytics

Saved outputs and execution metadata have been cleared. Optional odds requests default to off. Model logic is unchanged. See the project README for methods, limitations, and setup. Run in Google Colab to generate current results; this copy has not been executed end to end.


# Sunday screening update — v2

Run **Runtime → Run all** in Colab. Existing Drive locations, trained probability models, validation gates, and odds budget settings are preserved. `FETCH_ODDS` is still **False** as in the uploaded notebook; set it to True to use the odds integration (existing daily caches are reused).

Changes: teammate-role exclusions now consider direct position/workload exposure, while unknown injury identities and unresolved quarterbacks remain conservative. Team-wide context stays visible. A separate configurable scoring-opportunity screen removes low-volume players from review eligibility without altering their probabilities. Rankings include `opportunity_ok`, `review_reason`, and `injury_approval_reason`; the saved availability audit now reflects final eligibility. A summary explains exclusions after model selection.

The new screens are heuristics, not validated accuracy or profitability improvements. Injury-model approval is unchanged: historical gain alone is insufficient when the newest timestamped test is too old. All players remain visible in rankings. Final active status still needs confirmation. Local syntax and targeted screening tests passed; the full Colab data/Drive/API workflow has not been executed here.

# Nicolay's NFL TD Model — Sunday evaluation update

Run top to bottom in Google Colab with a CPU runtime. Google Drive keeps cached models, odds, and prediction history.

**Sunday setup:** set `SUNDAY_ONLY=True`, with blank `TARGET_DATE`, and `FETCH_ODDS=True` when you want sportsbook comparisons. The next unstarted Sunday slate is selected automatically. Set `SUNDAY_ONLY=False` for Thursday/Monday. This delivered file leaves paid odds fetching off until you enable it.

**Three versions, explicit evidence:**
1. Original baseline: its existing chronological model comparison remains intact.
2. Recent usage: adds last-appearance workload and changes relative to four-game averages. It is used only if predefined validation and later test checks pass.
3. Injury context: adds pregame teammate absences and an inferred expected QB. Historical reports must have timestamps before kickoff. Files without timestamps cannot be treated as known pregame information. An older injury benchmark is diagnostic; release requires a sufficiently recent test as well as accuracy checks.

**What this does not promise:** improved scores, profitable bets, or a pick every slate. Backtests use offensive participants and revised historical statistics. Weekly roster publication timestamps are not archived. The latest final historical season has already been inspected in prior versions, so it is temporal out-of-sample evidence, not a new untouched research holdout. Recent prospective results are the stronger check.

**No manual bet log:** Each run saves an immutable pregame forecast snapshot. Later runs automatically grade outcomes and show Brier scores and calibration. No offensive snap is excluded from conditional-model accuracy; it is not automatically called a sportsbook refund. No personal betting profit is inferred. Nothing runs while the notebook is closed.

Passing, return, defensive, and fumble-recovery touchdowns are outside the model. Check each book's anytime-TD settlement rules. Injury and historical-data downloads use no Odds API credits.


In [ ]:
# Install the data reader and modeling packages. The data reader version was tested.
%pip -q install nflreadpy==0.1.5 pandas pyarrow scikit-learn scipy joblib matplotlib jinja2

## 1. Settings — this is the main section to change

Dates and game times use **America/New_York**. A date range must stay within one NFL week. `SLATE_TEAMS` keeps any game involving those teams; use the exact abbreviations shown in the schedule. Set time limits to select an afternoon or evening slate.

Set `USE_GOOGLE_DRIVE=True` to keep your cache in Drive. Refresh current data before using rankings; historical seasons are downloaded once. To rebuild everything after a data correction, set `REBUILD_HISTORY=True` for one run, then change it back.

In [ ]:
SUNDAY_ONLY = False       # Next unstarted Sunday slate; False also permits Thursday/Monday.
TARGET_DATE = ""          # Example: "2026-09-27". Blank = next unstarted game day.
END_DATE = ""             # Blank = TARGET_DATE only. Example: "2026-09-28".
SLATE_TEAMS = []           # Example: ["KC", "BUF"]. Blank list = all games on the dates.
START_TIME_ET = "00:00"    # Example: "13:00".
END_TIME_ET = "23:59"      # Example: "16:30".

USE_GOOGLE_DRIVE = True   # Keep True to preserve saved odds and models between Colab sessions.
REFRESH_CURRENT_DATA = True
REBUILD_HISTORY = False
TOP_N = 30
FIRST_SEASON = 2020

# Optional additional exclusions. Automatic injury exclusions run regardless of this list.
EXCLUDE_PLAYERS = []

# Optional manual workload scenarios, only when you have a reason to change the role.
# Supported inputs: carries, targets, rush_inside5, rz_targets, offense_pct.
# These replace the player's recent 4-game averages ONLY. They are not injury forecasts.
ROLE_OVERRIDES = {}        # Example: {"Exact Player Name": {"carries": 18, "targets": 3}}

# Optional roster additions if a newly signed/promoted player is missing.
# Use an official GSIS player ID (not an ESPN ID) and the schedule's team abbreviation.
EXTRA_PLAYERS = []         # [{"player_id":"00-...", "player":"Name", "team":"KC", "position":"RB"}]

# ODDS BUDGET SETTINGS — independent of historical-data/model refresh settings.
FETCH_ODDS = False  # Enable only after configuring your own API key.
ODDS_REFRESH_TOKEN = ""   # Normal: leave blank. To refresh once, enter a NEW label, e.g. "pregame-1".
# Reusing the same label reuses that day's cache. Clear it after refreshing.
ODDS_CREDIT_RESERVE = 25  # Stop new paid requests before the account falls below this.
ODDS_MAX_CREDITS_PER_RUN = 20

# INJURY / ROLE CHECKS — free data; no Odds API credits.
INJURY_MAX_AGE_HOURS = 96  # Allows Friday reports for Monday; season/week must also match.
AUTO_ROLE_SCENARIOS = False
MIN_ROLE_COMPARISON_GAMES = 2
# Optional corrections use GSIS IDs AND an exact game_id so they expire with the game.
# An override does not certify an entire team's injury coverage.
AVAILABILITY_OVERRIDES = {}  # {'00-...': {'game_id':'2026_05_TB_DAL','status':'Out'}}

# Automatic empirical workload overrides are off. Trained model comparisons replace them.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import nflreadpy as nfl
import joblib
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from IPython.display import display

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    # Check for an existing mount rather than creating files in the mount point.
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Nicolays_NFL_TD_Model')
else:
    BASE = Path('/content/Nicolays_NFL_TD_Model') if Path('/content').exists() else Path.cwd() / 'td_model_work'
CACHE, OUTPUT = BASE / 'cache', BASE / 'rankings'
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
VERSION = 'td-v1.1'
NOW = pd.Timestamp.now(tz='America/New_York')
POSITIONS = ['QB', 'RB', 'WR', 'TE']
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)

def require(df, columns, source):
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f'{source} is missing columns: {sorted(missing)}. The upstream schema may have changed.')

def normalize_position(s):
    return s.replace({'LB/F':'FB'}).str.split('/').str[0].replace({'FB': 'RB', 'HB': 'RB'})

def cached_frame(name, loader, refresh=False):
    path = CACHE / f'{name}.parquet'
    if path.exists() and not refresh:
        return pd.read_parquet(path)
    for attempt in range(3):
        try:
            result = loader()
            df = result.to_pandas() if hasattr(result, 'to_pandas') else result
            if df.empty:
                raise ValueError(f'{name} returned no rows')
            temporary = path.with_suffix('.tmp.parquet')
            df.to_parquet(temporary, index=False)
            temporary.replace(path)
            return df
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f'Could not refresh {name}. No stale fallback was used. Retry later. Original error: {e}') from e
            time.sleep(2 * (attempt + 1))

# Clear the data reader's own cache when requesting a refresh; our historical cache stays.
if REFRESH_CURRENT_DATA or REBUILD_HISTORY:
    nfl.clear_cache()
print('Cache folder:', CACHE)

## 2. Select games and establish the information cutoff

This version freezes the input statistics at the **start of the selected NFL week**. That makes Thursday and Sunday predictions comparable and prevents partial-week updates from changing the historical test. A slate cannot include a game that has already started. Historical evaluation is handled separately below.

In [ ]:
schedule = cached_frame('schedule', lambda: pd.read_csv('https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv', low_memory=False), REFRESH_CURRENT_DATA)
require(schedule, ['game_id','season','week','game_type','gameday','gametime','home_team','away_team','home_score','away_score'], 'Schedule')
schedule = schedule.loc[schedule.game_type.eq('REG')].copy()
schedule['gameday'] = pd.to_datetime(schedule.gameday).dt.normalize()
schedule['kickoff'] = pd.to_datetime(schedule.gameday.dt.strftime('%Y-%m-%d') + ' ' + schedule.gametime.fillna('23:59')).dt.tz_localize('America/New_York')
if not TARGET_DATE:
    future = schedule.loc[(schedule.kickoff > NOW) & schedule.home_score.isna()]
    if SUNDAY_ONLY:
        future = future.loc[future.gameday.dt.dayofweek.eq(6)]
    if future.empty:
        raise ValueError('No upcoming regular-season schedule is available. Try again when the schedule is published.')
    target = future.gameday.min()
else:
    target = pd.Timestamp(TARGET_DATE).normalize()
end = pd.Timestamp(END_DATE).normalize() if END_DATE else target
if end < target:
    raise ValueError('END_DATE must be on or after TARGET_DATE.')
selected = schedule.loc[schedule.gameday.between(target, end)].copy()
if SLATE_TEAMS:
    selected = selected.loc[selected.home_team.isin(SLATE_TEAMS) | selected.away_team.isin(SLATE_TEAMS)]
selected = selected.loc[selected.gametime.fillna('').between(START_TIME_ET, END_TIME_ET)]
if selected.empty:
    display(schedule.loc[schedule.gameday.between(target-pd.Timedelta(days=3), end+pd.Timedelta(days=4)), ['gameday','gametime','away_team','home_team']])
    raise ValueError('No games match the settings. Choose one of the game dates shown above.')
if selected[['season','week']].drop_duplicates().shape[0] != 1:
    raise ValueError('Choose games from one NFL week at a time.')
if (selected.kickoff <= NOW).any() or selected.home_score.notna().any():
    raise ValueError('The slate includes a game that has started or finished. Select only upcoming games.')
SEASON, WEEK = int(selected.season.iloc[0]), int(selected.week.iloc[0])
if SEASON - FIRST_SEASON < 6:
    raise ValueError('Use at least six earlier seasons for training, calibration, validation and testing.')
week_start = schedule.loc[(schedule.season == SEASON) & (schedule.week == WEEK), 'gameday'].min()
print(f'Season {SEASON}, week {WEEK}. Only games before {week_start.date()} inform the model.')
display(selected[['game_id','gameday','gametime','away_team','home_team']].sort_values(['gameday','gametime']))

## 3. Build player history, including players with zero carries and targets

The historical sample starts from offensive snap counts, not just players with a box-score touch. Official player stats supply the touchdown outcomes; play-by-play supplies red-zone usage. Missing source games or unmatched player IDs trigger checks instead of silently treating missing records as zero performance. Full play-by-play is reduced one season at a time and the small results are cached.

In [ ]:
PBP_COLS = ['game_id','posteam','defteam','rusher_player_id','receiver_player_id',
            'rush_attempt','pass_attempt','rush_touchdown','pass_touchdown',
            'yardline_100','two_point_attempt','play_type','yards_gained','qb_kneel']
STAT_COLS = ['carries','targets','rushing_yards','receiving_yards','rushing_tds','receiving_tds']
USAGE_COLS = ['rush_inside5','rush_inside10','rz_carries','rz_targets','targets_inside10']
player_directory = cached_frame('player_directory', lambda: nfl.load_players(), REFRESH_CURRENT_DATA)
require(player_directory, ['pfr_id','gsis_id'], 'Player identity directory')

def season_tables(year):
    player_path = CACHE / f'{VERSION}_player_{year}.parquet'
    team_path = CACHE / f'{VERSION}_team_{year}.parquet'
    roster_path = CACHE / f'{VERSION}_roster_{year}.parquet'
    refresh = REBUILD_HISTORY or (year == SEASON and REFRESH_CURRENT_DATA)
    if all(p.exists() for p in [player_path, team_path, roster_path]) and not refresh:
        return pd.read_parquet(player_path), pd.read_parquet(team_path), pd.read_parquet(roster_path)
    print(f'Processing {year}...', flush=True)
    stats = cached_frame(f'stats_{year}', lambda: nfl.load_player_stats(year), refresh)
    snaps = cached_frame(f'snaps_{year}', lambda: nfl.load_snap_counts(year), refresh)
    rosters = cached_frame(f'rosters_{year}', lambda: nfl.load_rosters_weekly(year), refresh)
    require(stats, ['player_id','game_id','team','season_type'] + STAT_COLS, 'Player stats')
    require(snaps, ['game_id','pfr_player_id','position','team','offense_snaps','offense_pct','game_type'], 'Snap counts')
    require(rosters, ['gsis_id','pfr_id','full_name','position','team','week','status'], 'Weekly rosters')
    # Select columns in Polars before converting to pandas to keep memory use low.
    def read_pbp():
        raw = nfl.load_pbp(year)
        require(raw, PBP_COLS, 'Play-by-play')
        return raw.select(PBP_COLS)
    pbp = cached_frame(f'pbp_small_{year}', read_pbp, refresh)
    year_games = schedule.loc[(schedule.season == year) & (schedule.gameday < week_start) & schedule.home_score.notna()]
    game_ids = set(year_games.game_id)
    for label, frame in [('player stats',stats),('snap counts',snaps),('play-by-play',pbp)]:
        missing = game_ids - set(frame.game_id)
        if missing:
            raise ValueError(f'{year}: {label} is missing {len(missing)} completed games, e.g. {sorted(missing)[:3]}. Retry after the source updates.')
    stats = stats.loc[stats.game_id.isin(game_ids) & stats.season_type.eq('REG')].copy()
    snaps = snaps.loc[snaps.game_id.isin(game_ids) & (snaps.offense_snaps > 0)].copy()
    # Hybrid players can be labeled CB in PFR but WR in official player stats.
    # Use the position field (never whether they scored) to include these players.
    position_links = stats[['game_id','player_id','position']].merge(
        player_directory[['gsis_id','pfr_id']].dropna().drop_duplicates(),
        left_on='player_id',right_on='gsis_id',how='inner')
    position_links = position_links[['game_id','pfr_id','position']].drop_duplicates().rename(columns={'position':'stat_position'})
    snaps = snaps.merge(position_links,left_on=['game_id','pfr_player_id'],right_on=['game_id','pfr_id'],how='left',validate='many_to_one').drop(columns='pfr_id')
    official_position = normalize_position(snaps.stat_position)
    snaps.loc[official_position.isin(POSITIONS),'position'] = official_position
    snaps['position'] = normalize_position(snaps.position)
    snaps = snaps.loc[snaps.position.isin(POSITIONS)].copy()
    identity = pd.concat([player_directory[['pfr_id','gsis_id']], rosters[['pfr_id','gsis_id']]], ignore_index=True)
    identity = identity.loc[identity.gsis_id.notna() & identity.gsis_id.ne('') & identity.pfr_id.notna() & identity.pfr_id.ne('')].drop_duplicates()
    identity = identity.loc[identity.pfr_id.isin(snaps.pfr_player_id)]
    if identity.pfr_id.duplicated().any():
        raise ValueError('Ambiguous PFR-to-GSIS mapping. Inspect the roster source before proceeding.')
    panel = snaps.merge(identity, left_on='pfr_player_id', right_on='pfr_id', how='left', validate='many_to_one')
    # A few older players lack a PFR link even in the master directory. Resolve only
    # exact normalized names on the SAME season/team, and only if one GSIS ID matches.
    def name_key(values):
        return values.fillna('').str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.lower().str.replace(r'[^a-z]','',regex=True)
    named = rosters.loc[rosters.gsis_id.notna() & rosters.gsis_id.ne(''), ['full_name','team','gsis_id']].copy()
    named['name_key'] = name_key(named.full_name)
    unique_names = named.groupby(['name_key','team']).gsis_id.nunique()
    unique_names = unique_names.loc[unique_names.eq(1)].index
    named = named.set_index(['name_key','team']).loc[unique_names].reset_index().drop_duplicates(['name_key','team'])
    lookup = named.set_index(['name_key','team']).gsis_id.to_dict()
    keys = name_key(panel.player)
    missing_ids = panel.gsis_id.isna()
    panel.loc[missing_ids,'gsis_id'] = [lookup.get((keys.loc[i],panel.loc[i,'team'])) for i in panel.index[missing_ids]]
    # Verified cross-source alias: PIT TE Rodney Williams is "Rod Williams"
    # in nflverse rosters/stats. His master-directory PFR link is missing.
    # John Samuel Shenker is "John Shenker" in the roster/stat files.
    # Nathan Carter is "Nate Carter" in the roster/stat files.
    verified_aliases = {'WillRo08':'00-0037451', 'ShenJo00':'00-0038920', 'CartNa00':'00-0040547'}
    panel['gsis_id'] = panel.gsis_id.fillna(panel.pfr_player_id.map(verified_aliases))
    if panel.gsis_id.isna().any():
        bad = panel.loc[panel.gsis_id.isna(), ['player','pfr_player_id']].drop_duplicates()
        raise ValueError(f'Offensive players could not be mapped to GSIS IDs: {bad.head(8).to_dict("records")}')
    panel = panel.rename(columns={'gsis_id':'player_id'})
    panel = panel[['game_id','player_id','player','position','team','offense_pct','offense_snaps']]
    require(stats, STAT_COLS, 'Scoring statistics')
    scoring = stats[['game_id','player_id','team'] + STAT_COLS]
    if scoring.duplicated(['game_id','player_id','team']).any():
        raise ValueError('Duplicate player-game scoring rows.')
    # A positive TD scorer must appear in the offensive-snap sample.
    scorers = stats.loc[(stats.rushing_tds + stats.receiving_tds > 0) & normalize_position(stats.position).isin(POSITIONS)]
    missing_scorers = scorers.merge(panel[['game_id','player_id']], on=['game_id','player_id'], how='left', indicator=True)
    if missing_scorers._merge.eq('left_only').any():
        bad = missing_scorers.loc[missing_scorers._merge.eq('left_only'), ['game_id','player_display_name','player_id']]
        raise ValueError(f'Snap data is missing a touchdown scorer: {bad.to_dict("records")}')
    panel = panel.merge(scoring, on=['game_id','player_id','team'], how='left', validate='one_to_one')
    panel[STAT_COLS] = panel[STAT_COLS].fillna(0)
    pbp = pbp.loc[pbp.game_id.isin(game_ids) & pbp.two_point_attempt.fillna(0).ne(1) & pbp.play_type.ne('no_play')].copy()
    # Kneels should not inflate useful rushing opportunity.
    rush = pbp.loc[pbp.rush_attempt.eq(1) & pbp.qb_kneel.fillna(0).eq(0) & pbp.rusher_player_id.notna()].copy()
    rush['rush_inside5'] = rush.yardline_100.le(5).astype(int)
    rush['rush_inside10'] = rush.yardline_100.le(10).astype(int)
    rush['rz_carries'] = rush.yardline_100.le(20).astype(int)
    r = rush.groupby(['game_id','rusher_player_id'])[['rush_inside5','rush_inside10','rz_carries']].sum().reset_index().rename(columns={'rusher_player_id':'player_id'})
    target_plays = pbp.loc[pbp.pass_attempt.eq(1) & pbp.receiver_player_id.notna()].copy()
    target_plays['rz_targets'] = target_plays.yardline_100.le(20).astype(int)
    target_plays['targets_inside10'] = target_plays.yardline_100.le(10).astype(int)
    t = target_plays.groupby(['game_id','receiver_player_id'])[['rz_targets','targets_inside10']].sum().reset_index().rename(columns={'receiver_player_id':'player_id'})
    panel = panel.merge(r, on=['game_id','player_id'], how='left', validate='one_to_one').merge(t, on=['game_id','player_id'], how='left', validate='one_to_one')
    panel[USAGE_COLS] = panel[USAGE_COLS].fillna(0)
    totals = stats.groupby(['game_id','team'])[['carries','targets']].sum().rename(columns={'carries':'team_carries','targets':'team_targets'}).reset_index()
    panel = panel.merge(totals, on=['game_id','team'], how='left', validate='many_to_one')
    panel['carry_share'] = panel.carries.div(panel.team_carries.replace(0,np.nan)).fillna(0)
    panel['target_share'] = panel.targets.div(panel.team_targets.replace(0,np.nan)).fillna(0)
    panel['label'] = (panel.rushing_tds > 0).astype(int) + 2 * (panel.receiving_tds > 0).astype(int)
    panel = panel.merge(year_games[['game_id','season','week','gameday','home_team','away_team']], on='game_id', validate='many_to_one')
    panel['opponent'] = np.where(panel.team.eq(panel.home_team), panel.away_team, panel.home_team)
    panel['home'] = panel.team.eq(panel.home_team).astype(int)
    panel['pending'] = False
    live = pbp.loc[pbp.posteam.notna() & (pbp.rush_attempt.eq(1) | pbp.pass_attempt.eq(1))].copy()
    live['rz_play'] = live.yardline_100.le(20).astype(int)
    live['rush_yards'] = np.where(live.rush_attempt.eq(1), live.yards_gained.fillna(0), 0)
    live['pass_yards'] = np.where(live.pass_attempt.eq(1), live.yards_gained.fillna(0), 0)
    team = live.groupby(['game_id','posteam']).agg(
        rush_td=('rush_touchdown','sum'), pass_td=('pass_touchdown','sum'),
        rush_plays=('rush_attempt','sum'), pass_plays=('pass_attempt','sum'),
        rz_plays=('rz_play','sum'), rush_yards=('rush_yards','sum'), pass_yards=('pass_yards','sum')
    ).reset_index().rename(columns={'posteam':'team'})
    if len(team) != 2 * len(year_games):
        raise ValueError('Team play-by-play coverage is incomplete.')
    # Save unfiltered full weekly rosters so they can also select the upcoming slate.
    panel.to_parquet(player_path, index=False)
    team.to_parquet(team_path, index=False)
    rosters.to_parquet(roster_path, index=False)
    return panel, team, rosters

histories, team_histories = [], []
current_rosters = None
for year in range(FIRST_SEASON, SEASON + 1):
    # Week 1 has no current-season stats yet; only current rosters are needed.
    current_games = schedule.loc[(schedule.season == year) & (schedule.gameday < week_start) & schedule.home_score.notna()]
    if year == SEASON and current_games.empty:
        current_rosters = cached_frame(f'rosters_{year}', lambda: nfl.load_rosters_weekly(year), REFRESH_CURRENT_DATA)
        continue
    players_year, teams_year, rosters_year = season_tables(year)
    histories.append(players_year)
    team_histories.append(teams_year)
    if year == SEASON:
        current_rosters = rosters_year
history = pd.concat(histories, ignore_index=True)
history = history.loc[history.gameday < week_start].copy()
team_history = pd.concat(team_histories, ignore_index=True)
team_history = team_history.loc[team_history.game_id.isin(history.game_id.unique())].copy()
print(f'{len(history):,} historical player-games. Latest game used: {history.gameday.max().date()}.')

## 4. Upcoming players and strictly lagged features

The latest available roster at or before the requested week is used. `ACT` means listed on the active roster, **not** confirmed game-day participation. Backups remain in the output with role warnings. No-history players use training-sample medians and position information; their forecasts are especially uncertain.

In [ ]:
# Screening rules only: these do not change fitted model probabilities.
# Thresholds are transparent heuristics, not backtested profit-optimal cutoffs.
MIN_REVIEW_RB_OPPORTUNITIES = 5.0
MIN_REVIEW_RECEIVER_TARGETS = 3.0
MIN_REVIEW_RZ_OPPORTUNITIES = 0.5
MIN_REVIEW_QB_CARRIES = 3.0
MIN_REVIEW_QB_INSIDE5 = 0.25

def teammate_role_relevant(player, teammate_position):
    """Direct workload exposure; QB changes affect everyone. Unknowns stay blocked.
    WR/TE changes affect pass catchers and receiving backs; RB changes affect RBs.
    Indirect team effects remain visible in role_context even when not blocking.
    """
    own = str(player.position)
    other = str(teammate_position)
    if other not in {'QB', 'RB', 'FB', 'WR', 'TE'}:
        return True
    if own == 'QB' or other == 'QB':
        return True
    if other in {'RB', 'FB'}:
        return own in {'RB', 'FB'}
    targets = pd.to_numeric(player.get('targets_last4', np.nan), errors='coerce')
    return own in {'WR', 'TE'} or (own in {'RB', 'FB'} and (pd.isna(targets) or targets >= 3))

def opportunity_screen(row):
    def value(name):
        x = pd.to_numeric(row.get(name, np.nan), errors='coerce')
        return float(x) if pd.notna(x) and np.isfinite(x) else 0.0
    position = str(row.position)
    carries, targets = value('carries_last4'), value('targets_last4')
    goal_line = value('rush_inside5_last4')
    rz = value('rz_targets_last4')
    if position in {'RB', 'FB'}:
        ok = carries + targets >= MIN_REVIEW_RB_OPPORTUNITIES or goal_line >= MIN_REVIEW_RZ_OPPORTUNITIES
    elif position in {'WR', 'TE'}:
        ok = targets >= MIN_REVIEW_RECEIVER_TARGETS or rz >= MIN_REVIEW_RZ_OPPORTUNITIES
    elif position == 'QB':
        ok = carries >= MIN_REVIEW_QB_CARRIES and goal_line >= MIN_REVIEW_QB_INSIDE5
    else:
        ok = False
    return bool(ok)


In [ ]:
require(current_rosters, ['week','gsis_id','full_name','position','team','status'], 'Current rosters')
roster = current_rosters.loc[current_rosters.week.le(WEEK)].copy()
if roster.empty:
    raise ValueError('No roster snapshot is available for this week yet.')
roster_week = int(roster.week.max())
roster = roster.loc[roster.week.eq(roster_week)].copy()
roster['position'] = normalize_position(roster.position)
roster = roster.loc[roster.position.isin(POSITIONS) & roster.gsis_id.notna() & roster.gsis_id.ne('')]
roster = roster.rename(columns={'gsis_id':'player_id','full_name':'player'})
roster = roster[['player_id','player','position','team','status']].rename(columns={'status':'roster_status'}).drop_duplicates()
if EXTRA_PLAYERS:
    extra = pd.DataFrame(EXTRA_PLAYERS)
    extra['roster_status'] = 'ACT'
    require(extra, ['player_id','player','team','position'], 'EXTRA_PLAYERS')
    if not extra.position.isin(POSITIONS).all() or extra.player_id.duplicated().any():
        raise ValueError('EXTRA_PLAYERS must have unique IDs and QB/RB/WR/TE positions.')
    roster = pd.concat([roster.loc[~roster.player_id.isin(extra.player_id)], extra], ignore_index=True)
if roster.player_id.duplicated().any():
    raise ValueError('A player has multiple current roster entries; resolve his current team before ranking.')
game_teams = []
for side, other in [('home','away'),('away','home')]:
    g = selected[['game_id','season','week','gameday','gametime',f'{side}_team',f'{other}_team']].copy()
    g = g.rename(columns={f'{side}_team':'team', f'{other}_team':'opponent'})
    g['home'] = int(side == 'home')
    game_teams.append(g)
game_teams = pd.concat(game_teams, ignore_index=True)
pending = roster.merge(game_teams, on='team', how='inner', validate='many_to_one')
if pending.empty or set(game_teams.team) - set(pending.team):
    raise ValueError('Roster coverage is incomplete for the selected teams.')
unknown_exclusions = set(EXCLUDE_PLAYERS) - set(pending.player)
if unknown_exclusions:
    print('Names not found in this slate:', sorted(unknown_exclusions))
# Retain excluded players until injury context has been evaluated for their teammates.
if pending.empty:
    raise ValueError('All players were excluded.')
pending['pending'] = True
pending['label'] = np.nan
print(f'Using roster week {roster_week}; requested week {WEEK}. Checking injuries next.')

PLAYER_METRICS = STAT_COLS + USAGE_COLS + ['offense_pct','carry_share','target_share']
TEAM_METRICS = ['rush_td','pass_td','rush_plays','pass_plays','rz_plays','rush_yards','pass_yards']

def make_features(past, upcoming, team_past):
    data = pd.concat([past, upcoming], ignore_index=True).sort_values(['player_id','gameday','game_id']).reset_index(drop=True)
    if data.duplicated(['player_id','game_id']).any():
        raise ValueError('Duplicate player-game rows.')
    for metric in PLAYER_METRICS:
        for window in [4,12]:
            data[f'{metric}_last{window}'] = data.groupby('player_id', sort=False)[metric].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    data['prior_games'] = data.groupby('player_id').cumcount().clip(upper=40)
    previous_date = data.groupby('player_id').gameday.shift(1)
    data['days_since_played'] = (data.gameday - previous_date).dt.days.clip(upper=365)
    previous_team = data.groupby('player_id').team.shift(1)
    data['new_team'] = (previous_team.notna() & previous_team.ne(data.team)).astype(int)
    data['no_history'] = data.prior_games.eq(0).astype(int)
    # Make one row per team-game, then shift each team's own results.
    games = data[['game_id','season','week','gameday','team','opponent']].drop_duplicates()
    team = games.merge(team_past, on=['game_id','team'], how='left', validate='one_to_one')
    allowed = team_past.rename(columns={'team':'opponent', **{m:'allowed_'+m for m in TEAM_METRICS}})
    team = team.merge(allowed, on=['game_id','opponent'], how='left', validate='one_to_one').sort_values(['team','gameday'])
    lag_cols = []
    for metric in TEAM_METRICS + ['allowed_'+m for m in TEAM_METRICS]:
        col = metric + '_last8'
        team[col] = team.groupby('team')[metric].transform(lambda s: s.shift(1).rolling(8,min_periods=1).mean())
        lag_cols.append(col)
    team['team_rest'] = team.groupby('team').gameday.diff().dt.days.clip(upper=30)
    offense_cols = [m+'_last8' for m in TEAM_METRICS]
    data = data.merge(team[['game_id','team','team_rest']+offense_cols], on=['game_id','team'], how='left', validate='many_to_one')
    defense_cols = ['allowed_'+m+'_last8' for m in TEAM_METRICS]
    defense = team[['game_id','team']+defense_cols].rename(columns={'team':'opponent'})
    data = data.merge(defense, on=['game_id','opponent'], how='left', validate='many_to_one')
    features = [f'{m}_last{w}' for m in PLAYER_METRICS for w in [4,12]] + ['prior_games','days_since_played','new_team','no_history','home','week','team_rest'] + offense_cols + defense_cols
    return data, features

data, NUMERIC = make_features(history, pending, team_history)
FEATURES = NUMERIC + ['position']
training = data.loc[~data.pending].copy()
prediction_rows = data.loc[data.pending].copy()
# Injury checks are free nflverse data requests, independent of the Odds API.
# No inferred probability of playing and no arbitrary transfer of touches.
def standard_status(value):
    text = '' if pd.isna(value) else str(value).strip().lower()
    lookup = {'out':'OUT', 'doubtful':'DOUBTFUL', 'questionable':'QUESTIONABLE',
              'probable':'PROBABLE', 'active':'ACTIVE', 'inactive':'INACTIVE',
              'ir':'IR', 'injured reserve':'IR', 'reserve/injured':'IR',
              'pup':'PUP', 'suspended':'SUSPENDED', 'not listed':'NOT_LISTED',
              '':'NO_DESIGNATION', 'nan':'NO_DESIGNATION', 'none':'NO_DESIGNATION'}
    return lookup.get(text, 'UNKNOWN')

def prepare_injury_report(raw, season, week, now, snapshot_timestamp=None):
    required = ['season','week','team','gsis_id','report_status']
    require(raw, required, 'Injury report')
    report = raw.loc[pd.to_numeric(raw.season, errors='coerce').eq(season) &
                     pd.to_numeric(raw.week, errors='coerce').eq(week)].copy()
    if 'season_type' in report:
        report = report.loc[report.season_type.astype(str).str.upper().eq('REG')]
    if 'date_modified' in report:
        report['updated_at'] = pd.to_datetime(report.date_modified, utc=True, errors='coerce')
        report['timestamp_basis'] = 'per_player_report'
    else:
        stamp = pd.to_datetime(snapshot_timestamp, utc=True, errors='coerce')
        if pd.isna(stamp):
            raise ValueError('Injury source has neither row timestamps nor a publication timestamp.')
        report['updated_at'] = stamp
        report['timestamp_basis'] = 'source_file_last_modified_NOT_individual_report_time'
    age = (now.tz_convert('UTC') - report.updated_at).dt.total_seconds()/3600
    report = report.loc[age.between(-0.1, INJURY_MAX_AGE_HOURS)].copy()
    report['injury_status'] = report.report_status.map(standard_status)
    # Team aliases follow the schedule/nflverse abbreviations.
    report['team'] = report.team.replace({'LA':'LAR', 'WSH':'WAS'})
    report = report.sort_values('updated_at').drop_duplicates(['team','gsis_id'], keep='last')
    return report

def fetch_injury_report():
    try:
        # Always refresh injuries, even when the model/history cache is reused.
        import io, urllib.request
        url = f'https://github.com/nflverse/nflverse-data/releases/download/injuries/injuries_{SEASON}.csv'
        with urllib.request.urlopen(url, timeout=30) as response:
            published = response.headers.get('Last-Modified')
            raw = pd.read_csv(io.BytesIO(response.read()), low_memory=False)
        report = prepare_injury_report(raw, SEASON, WEEK,
            pd.Timestamp.now(tz='America/New_York'), snapshot_timestamp=published)
        if report.empty:
            raise ValueError('No fresh injury rows for the selected season/week.')
        return report, 'nflverse'
    except Exception as err:
        print('INJURY COVERAGE UNAVAILABLE:', type(err).__name__, str(err)[:240])
        print('Rankings remain provisional. No value candidates will be endorsed without coverage.')
        return pd.DataFrame(columns=['team','gsis_id','injury_status','updated_at','timestamp_basis']), 'unavailable'

def apply_injuries_and_roles(rows, past, roster_history, report, now):
    rows = rows.copy()
    rows['injury_status'] = 'UNKNOWN'
    rows['injury_checked_at_utc'] = now.tz_convert('UTC').isoformat()
    rows['injury_report_updated_at_utc'] = ''
    rows['injury_source'] = 'unavailable'
    rows['injury_timestamp_basis'] = 'unavailable'
    rows['injury_coverage_ok'] = False
    rows['role_adjustment_source'] = 'recent_usage_only'
    rows['role_adjustment_games'] = 0
    rows['role_context'] = ''
    rows['role_uncertain'] = False
    rows['direct_role_change'] = False
    rows['role_adjusted'] = False
    rows['excluded'] = False
    blocked_statuses = {'OUT','IR','PUP','SUSPENDED','INACTIVE'}
    uncertain_statuses = {'QUESTIONABLE','DOUBTFUL','UNKNOWN'}
    covered = set(report.team)
    # Missing GSIS IDs must not silently become 'healthy' unmatched players.
    broken_teams = set(report.loc[report.gsis_id.isna() | report.gsis_id.astype(str).eq(''), 'team'])
    broken_teams |= set(report.loc[report.injury_status.eq('UNKNOWN'), 'team'])
    for idx, row in rows.iterrows():
        if row.team in covered and row.team not in broken_teams:
            rows.at[idx,'injury_coverage_ok'] = True
            rows.at[idx,'injury_status'] = 'NOT_LISTED'
            rows.at[idx,'injury_source'] = 'nflverse'
            rows.at[idx,'injury_timestamp_basis'] = report.loc[report.team.eq(row.team),'timestamp_basis'].iloc[-1]
            rows.at[idx,'injury_report_updated_at_utc'] = report.loc[report.team.eq(row.team),'updated_at'].max().isoformat()
        match = report.loc[report.team.eq(row.team) & report.gsis_id.eq(row.player_id)]
        if len(match):
            record = match.iloc[-1]
            rows.at[idx,'injury_status'] = record.injury_status
            rows.at[idx,'injury_report_updated_at_utc'] = record.updated_at.isoformat()
            rows.at[idx,'injury_source'] = 'nflverse'
            rows.at[idx,'injury_timestamp_basis'] = record.timestamp_basis
        if str(row.get('roster_status','ACT')) != 'ACT':
            rows.at[idx,'injury_status'] = 'INACTIVE'
            rows.at[idx,'injury_source'] = 'weekly_roster'
        # Overrides require an explicit game identity; they cannot carry into next week's slate.
        override = AVAILABILITY_OVERRIDES.get(row.player_id)
        if override is not None and override.get('game_id') == row.game_id:
            status = standard_status(override.get('status'))
            if status in {'UNKNOWN','NO_DESIGNATION','NOT_LISTED'}:
                raise ValueError('Availability override needs an explicit active/out/questionable status.')
            rows.at[idx,'injury_status'] = status
            rows.at[idx,'injury_source'] = 'manual_game_override'
        rows.at[idx,'excluded'] = rows.at[idx,'injury_status'] in blocked_statuses or row.player in EXCLUDE_PLAYERS
    # Snapshot every original 4-game feature before scenario changes.
    role_metrics = [m for m in PLAYER_METRICS if m not in {'rushing_tds','receiving_tds'}]
    for metric in role_metrics:
        rows['baseline_'+metric+'_last4'] = rows[metric+'_last4']
    for team, group in rows.groupby('team'):
        team_past = past.loc[past.team.eq(team) & past.season.eq(SEASON)].copy()
        recent_ids = team_past[['game_id','gameday']].drop_duplicates().sort_values('gameday').tail(4).game_id
        # Detect unavailable players even if a reserve player fell out of today's roster.
        latest = team_past.loc[team_past.game_id.isin(recent_ids)].groupby('player_id').agg(
            offense_pct=('offense_pct','mean'), carries=('carries','mean'), targets=('targets','mean'))
        meaningful = set(latest.index[(latest.offense_pct >= .35) | (latest.carries >= 5) | (latest.targets >= 3)])
        no_history = group.loc[group.prior_games.eq(0), 'player_id']
        # No-history reserves are not evidence of a material loss of current workload.
        for pid, rec in group.iterrows():
            if rec.player_id in set(latest.index) and (rec.offense_pct_last4 >= .35 or rec.carries_last4 >= 5 or rec.targets_last4 >= 3):
                meaningful.add(rec.player_id)
        team_report = report.loc[report.team.eq(team)]
        absent = set(group.loc[group.excluded,'player_id'])
        absent |= set(team_report.loc[team_report.injury_status.isin(blocked_statuses),'gsis_id'].dropna())
        uncertain = set(group.loc[group.injury_status.isin(uncertain_statuses),'player_id'])
        uncertain |= set(team_report.loc[team_report.injury_status.isin(uncertain_statuses),'gsis_id'].dropna())
        relevant_out = absent & meaningful
        relevant_uncertain = (uncertain & meaningful) - absent
        # Returning starters can also shrink a replacement's role. Flag that change.
        if len(recent_ids):
            last_game = recent_ids.iloc[-1]
            last_participants = set(team_past.loc[team_past.game_id.eq(last_game),'player_id'])
            returning = (set(group.loc[~group.excluded,'player_id']) & meaningful) - last_participants
            returning -= set(no_history)
            relevant_uncertain |= returning
        # Unknown offensive injury identities require review, even if not in today's roster.
        unseen = team_report.loc[~team_report.gsis_id.isin(set(group.player_id) | set(team_past.player_id))]
        if 'position' in unseen:
            unseen = unseen.loc[unseen.position.isin(POSITIONS)]
        unresolved = len(unseen.loc[unseen.injury_status.isin(blocked_statuses | uncertain_statuses)]) > 0
        if not relevant_out and not relevant_uncertain and not unresolved:
            continue
        context = 'Out IDs: ' + ', '.join(sorted(relevant_out)) + '; uncertain IDs: ' + ', '.join(sorted(relevant_uncertain))
        if unresolved:
            context += '; offensive injury identity/history unresolved'
        rows.loc[group.index,'role_context'] = context
        # Keep team context for everyone, but block only direct workload exposure.
        positions = team_past.sort_values('gameday').groupby('player_id').position.last().to_dict()
        positions.update(group.set_index('player_id').position.to_dict())
        affected_ids = relevant_out | relevant_uncertain
        affected = group.apply(lambda player: unresolved or any(
            pid == player.player_id or teammate_role_relevant(player, positions.get(pid, 'UNKNOWN'))
            for pid in affected_ids), axis=1)
        affected_index = group.index[affected]
        rows.loc[affected_index,'role_uncertain'] = True
        rows.loc[affected_index,'direct_role_change'] = True
        # Do not assume questionable teammates sit, or project a new role without evidence.
        if relevant_uncertain or unresolved or not AUTO_ROLE_SCENARIOS or not relevant_out:
            continue
        # Require the absent players to have belonged to this same team in each comparison week.
        known_roster = roster_history.loc[roster_history.team.eq(team)]
        usable_games = []
        for game in team_past[['game_id','week','gameday']].drop_duplicates().sort_values('gameday').tail(12).itertuples():
            ids_on_roster = set(known_roster.loc[known_roster.week.eq(game.week),'gsis_id'])
            ids_played = set(team_past.loc[team_past.game_id.eq(game.game_id),'player_id'])
            if relevant_out.issubset(ids_on_roster) and not relevant_out.intersection(ids_played):
                usable_games.append(game.game_id)
        for idx, row in group.loc[~group.excluded & affected].iterrows():
            similar = team_past.loc[team_past.game_id.isin(usable_games) & team_past.player_id.eq(row.player_id)].sort_values('gameday').tail(4)
            if len(similar) < MIN_ROLE_COMPARISON_GAMES:
                continue
            # Empirical scenario only: do not call this a validated injury model.
            for metric in role_metrics:
                value = similar[metric].mean()
                if pd.notna(value):
                    rows.at[idx,metric+'_last4'] = float(value)
            changed = any(not np.isclose(rows.at[idx,m+'_last4'], row[m+'_last4'], equal_nan=True) for m in role_metrics)
            rows.at[idx,'role_adjusted'] = changed
            rows.at[idx,'role_adjustment_source'] = ('same_team_absence_scenario_UNVALIDATED' if changed else 'recent_usage_already_matches_absence_scenario')
            rows.at[idx,'role_adjustment_games'] = len(similar)
    # A low-usage/returning player can have uncertain role without a currently injured teammate.
    rows.loc[rows.days_since_played.gt(21) | rows.new_team.eq(1), 'role_uncertain'] = True
    rows.loc[rows.position.eq('QB') & rows.offense_pct_last4.fillna(0).lt(.5), 'role_uncertain'] = True
    rows['review_eligible'] = (rows.injury_coverage_ok & ~rows.excluded & ~rows.role_uncertain &
        rows.injury_status.isin(['ACTIVE','NOT_LISTED','NO_DESIGNATION','PROBABLE']) &
        rows.offense_pct_last4.fillna(0).ge(.15) & rows.prior_games.ge(4))
    if roster_week < WEEK:
        rows['review_eligible'] = False
    return rows

injury_report, injury_feed_source = fetch_injury_report()
prediction_rows = apply_injuries_and_roles(prediction_rows, history, current_rosters, injury_report, pd.Timestamp.now(tz='America/New_York'))
availability_audit = prediction_rows[['player_id','player','team','game_id','injury_status',
    'injury_source','injury_timestamp_basis','injury_checked_at_utc','injury_report_updated_at_utc','injury_coverage_ok',
    'excluded','role_adjusted','role_adjustment_source','role_adjustment_games','role_context',
    'role_uncertain','review_eligible']].copy()
print('Availability and role audit (not a confirmed game-day active list):')
display(availability_audit)
prediction_rows = prediction_rows.loc[~prediction_rows.excluded].copy()
if prediction_rows.empty:
    raise ValueError('No eligible offensive participants remain after availability exclusions.')

allowed_overrides = {'carries','targets','rush_inside5','rz_targets','offense_pct'}
for player, changes in ROLE_OVERRIDES.items():
    mask = prediction_rows.player.eq(player)
    if mask.sum() != 1:
        raise ValueError(f'Role override must match exactly one slate player: {player}')
    for field, value in changes.items():
        if field not in allowed_overrides or not np.isfinite(value) or value < 0 or (field == 'offense_pct' and value > 1):
            raise ValueError(f'Invalid role override: {field}={value}')
        prediction_rows.loc[mask, field+'_last4'] = float(value)
    prediction_rows.loc[mask, 'role_adjusted'] = True
    prediction_rows.loc[mask, 'role_adjustment_source'] = 'manual_scenario_UNVALIDATED'
    prediction_rows.loc[mask, 'role_uncertain'] = True
    prediction_rows.loc[mask, 'review_eligible'] = False
assert not any(c in FEATURES for c in ['rushing_tds','receiving_tds','label','offense_pct'])
assert training.gameday.max() < prediction_rows.gameday.min()
print(f'{len(FEATURES)} pregame features; {len(prediction_rows)} slate players.')
# Reflect manual scenarios in the saved audit too.
for column in ['role_adjusted','role_adjustment_source','role_uncertain','review_eligible']:
    lookup = prediction_rows.set_index('player_id')[column]
    mask = availability_audit.player_id.isin(lookup.index)
    availability_audit.loc[mask,column] = availability_audit.loc[mask,'player_id'].map(lookup)


## Pregame injury context and source coverage
Historical files without usable pregame timestamps are excluded from injury features.

In [ ]:
# Pregame injury features. Archive report timestamps must precede each kickoff.
INJURY_FEATURE_VERSION = 'pregame-context-v1'
CONTEXT_FEATURES = ['ctx_coverage','ctx_own_questionable','ctx_own_doubtful',
    'ctx_out_carry_share','ctx_out_target_share','ctx_same_position_out_snap',
    'ctx_ol_out','ctx_expected_qb_changed','ctx_expected_qb_experience',
    'ctx_expected_qb_snap','ctx_is_expected_qb']

def load_injury_archives(years):
    import io, urllib.request
    frames, records = [], []
    archive_folder = CACHE / 'injury_archives'
    archive_folder.mkdir(exist_ok=True)
    for year in years:
        path = archive_folder / f'injuries_{year}.csv'
        try:
            if path.exists() and not REBUILD_HISTORY:
                frame = pd.read_csv(path, low_memory=False)
            else:
                url = f'https://github.com/nflverse/nflverse-data/releases/download/injuries/injuries_{year}.csv'
                with urllib.request.urlopen(url, timeout=30) as response:
                    frame = pd.read_csv(io.BytesIO(response.read()), low_memory=False)
                frame.to_csv(path,index=False)
            require(frame,['season','week','team','gsis_id','report_status'],'Archived injuries')
            kind = 'season_type' if 'season_type' in frame else 'game_type'
            if kind not in frame:
                raise ValueError('No regular/postseason identifier')
            frame = frame.loc[frame[kind].eq('REG')].copy()
            if 'date_modified' not in frame:
                records.append({'season':year,'rows':len(frame),'usable':False,'reason':'No per-report timestamps; excluded from pregame features'})
                continue
            frame['updated_at'] = pd.to_datetime(frame.date_modified, utc=True, errors='coerce')
            frame['injury_status'] = frame.report_status.map(standard_status)
            frame['team'] = frame.team.replace({'LA':'LAR','WSH':'WAS'})
            frames.append(frame)
            records.append({'season':year,'rows':len(frame),'usable':True,'reason':'Report timestamps will also be checked against kickoff'})
        except Exception as err:
            records.append({'season':year,'rows':0,'usable':False,'reason':f'{type(err).__name__}: {str(err)[:120]}'})
    columns=['season','week','team','gsis_id','injury_status','updated_at']
    return (pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=columns)), pd.DataFrame(records)

def pregame_context(players, past, reports, schedule_frame, roster_frames, live=False, asof=None):
    """No current-game snaps, outcomes or actual starting-QB fields are features.
    Teammate metrics use previous appearances before the START of the NFL week.
    """
    output = pd.DataFrame(0.,index=players.index,columns=CONTEXT_FEATURES)
    output['ctx_expected_qb_id'] = ''
    output['ctx_reason'] = 'INJURY_CONTEXT_UNKNOWN'
    output['ctx_material_absence'] = False
    output['ctx_qb_known'] = False
    game_lookup = schedule_frame.set_index('game_id')
    cutoff_lookup = schedule_frame.groupby(['season','week']).gameday.min().to_dict()
    reports_by_key = {k:g for k,g in reports.groupby(['season','week','team'])}
    roster_lookup = {k:g for k,g in roster_frames.groupby(['season','week','team'])}
    asof = pd.Timestamp.now(tz='UTC') if asof is None else pd.Timestamp(asof).tz_convert('UTC')
    for (gid,team), group in players.groupby(['game_id','team']):
        game = game_lookup.loc[gid]
        season, week = int(game.season), int(game.week)
        kickoff = pd.Timestamp(game.kickoff).tz_convert('UTC')
        deadline = min(kickoff,asof) if live else kickoff
        cutoff = cutoff_lookup[(season,week)]
        report = reports_by_key.get((season,week,team), pd.DataFrame())
        if report.empty:
            continue
        age = (deadline-report.updated_at).dt.total_seconds()/3600
        report = report.loc[age.between(0,96)].sort_values('updated_at').drop_duplicates('gsis_id',keep='last')
        if report.empty or report.gsis_id.isna().any() or report.injury_status.eq('UNKNOWN').any():
            continue
        roster = roster_lookup.get((season,week,team),pd.DataFrame())
        if roster.empty:
            continue
        team_past = past.loc[past.team.eq(team) & past.gameday.lt(cutoff) & past.season.ge(season-1)].sort_values('gameday')
        # Only current-roster players can be projected starters. Missing current roster means unknown.
        current_ids = set(roster.gsis_id.dropna())
        team_past = team_past.loc[team_past.player_id.isin(current_ids)]
        if team_past.empty:
            continue
        last = team_past.groupby('player_id').tail(4)
        summary = last.groupby('player_id').agg(carry=('carry_share','mean'),target=('target_share','mean'),snap=('offense_pct','mean'),position=('position','last'),last_played=('gameday','max'))
        summary['experience'] = team_past.groupby('player_id').size().clip(upper=40)
        # Longstanding absences are not assumed to remove touches from the recent offense.
        recent_games = team_past[['game_id','gameday']].drop_duplicates().sort_values('gameday').tail(4).game_id
        recently_played = set(team_past.loc[team_past.game_id.isin(recent_games),'player_id'])
        status = report.set_index('gsis_id').injury_status.to_dict()
        reserve = set(roster.loc[~roster.status.eq('ACT'),'gsis_id']) if 'status' in roster else set()
        out_ids = {pid for pid,st in status.items() if st in {'OUT','IR','INACTIVE','SUSPENDED','PUP'}} | reserve
        out = summary.loc[summary.index.isin(out_ids & recently_played)]
        qb = summary.loc[summary.position.eq('QB')].sort_values(['snap','last_played'],ascending=False)
        expected = qb.loc[~qb.index.isin(out_ids)]
        qb_known = not expected.empty
        expected_id = str(expected.index[0]) if qb_known else ''
        expected_status = status.get(expected_id,'NOT_LISTED')
        if expected_status in {'QUESTIONABLE','DOUBTFUL','UNKNOWN'}:
            qb_known=False
        unknown_out = report.loc[report.injury_status.isin(['OUT','DOUBTFUL']) & ~report.gsis_id.isin(summary.index)]
        if 'position' in unknown_out:
            unknown_out=unknown_out.loc[unknown_out.position.eq('QB')]
        if len(unknown_out):
            qb_known=False
        unresolved = report.loc[report.injury_status.isin(['QUESTIONABLE','DOUBTFUL']) & report.gsis_id.isin(summary.index)]
        material_ids = set(summary.index[(summary.snap>=.35) | (summary.carry>=.2) | (summary.target>=.15)])
        unresolved_material=bool(set(unresolved.gsis_id)&material_ids)
        ix=group.index
        output.loc[ix,'ctx_coverage']=1.
        output.loc[ix,'ctx_out_carry_share']=min(1.,out.carry.sum())
        output.loc[ix,'ctx_out_target_share']=min(1.,out.target.sum())
        output.loc[ix,'ctx_expected_qb_id']=expected_id
        output.loc[ix,'ctx_qb_known']=qb_known
        output.loc[ix,'ctx_expected_qb_changed']=float(qb_known and expected_id!=str(qb.index[0]))
        output.loc[ix,'ctx_expected_qb_experience']=float(expected.iloc[0].experience) if qb_known else 0.
        output.loc[ix,'ctx_expected_qb_snap']=float(expected.iloc[0].snap) if qb_known else 0.
        ol=report.loc[report.injury_status.eq('OUT')]
        output.loc[ix,'ctx_ol_out']=int(ol.position.isin(['C','G','T','OT','OG','OL']).sum()) if 'position' in ol else 0
        output.loc[ix,'ctx_material_absence']=bool(set(out.index)&material_ids)
        output.loc[ix,'ctx_reason']=('QUESTIONABLE_TEAM_ROLE' if unresolved_material else ('EXPECTED_QB_UNRESOLVED' if not qb_known else 'PREGAME_CONTEXT_AVAILABLE'))
        for idx,row in group.iterrows():
            st=status.get(row.player_id,'NOT_LISTED')
            output.at[idx,'ctx_own_questionable']=float(st=='QUESTIONABLE')
            output.at[idx,'ctx_own_doubtful']=float(st=='DOUBTFUL')
            output.at[idx,'ctx_same_position_out_snap']=out.loc[out.position.eq(row.position),'snap'].sum()
            output.at[idx,'ctx_is_expected_qb']=float(str(row.player_id)==expected_id)
    return output

# These source files are free. No odds request occurs in this section.
archives, archive_status = load_injury_archives(range(FIRST_SEASON, SEASON))
print('Historical injury source coverage:'); display(archive_status)
roster_parts=[]
for year in range(FIRST_SEASON,SEASON+1):
    path=CACHE / f'{VERSION}_roster_{year}.parquet'
    if year==SEASON:
        rr=current_rosters.copy()
    elif path.exists():
        rr=pd.read_parquet(path)
    else:
        continue
    rr['season']=year
    roster_parts.append(rr)
context_rosters=pd.concat(roster_parts,ignore_index=True)
training_context=pregame_context(training,history,archives,schedule,context_rosters)
live_report=injury_report.copy()
live_report['season']=SEASON;live_report['week']=WEEK
live_context=pregame_context(prediction_rows,history,live_report,schedule,context_rosters,live=True)
for col in training_context:
    training[col]=training_context[col]
    prediction_rows[col]=live_context[col]
context_coverage=training.groupby('season').agg(players=('player_id','size'),covered=('ctx_coverage','sum'),games=('game_id','nunique'))
context_coverage['covered_fraction']=context_coverage.covered/context_coverage.players
print('Usable pregame injury coverage:');display(context_coverage.round(3))


## 5. Honest backtest and model comparison

For a 2026 slate: fit on **2020–2022**, calibrate on **2023**, select the model on **2024**, then report a **2025 temporal test** (already inspected during earlier development; not a fresh untouched holdout). In each later season, features can use earlier completed games but never the current game's outcome. The fitted coefficients/trees stay frozen during the held-out test.

Temperature calibration adjusts confidence using an independent season, preserving the four outcome probabilities. Selection uses the combined-TD **Brier score** (lower is better). We also report log loss, AUC, and calibration bins. The position-only baseline tests whether detailed features improve on a simple benchmark. A good ranking or historical score does not establish profitability.

In [ ]:
CAL_YEAR, VALID_YEAR, TEST_YEAR = SEASON-3, SEASON-2, SEASON-1
fit_data = training.loc[training.season < CAL_YEAR]
cal_data = training.loc[training.season == CAL_YEAR]
valid_data = training.loc[training.season == VALID_YEAR]
test_data = training.loc[training.season == TEST_YEAR]
for label, part in [('fit',fit_data),('calibration',cal_data),('validation',valid_data),('test',test_data)]:
    if len(part) < 1000 or part.label.nunique() < 4:
        raise ValueError(f'Insufficient data for {label}: {len(part)} rows, {part.label.nunique()} outcome classes.')

def new_model(kind):
    numeric = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler())
    preprocess = ColumnTransformer([('numbers', numeric, NUMERIC), ('position', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['position'])])
    if kind == 'Logistic':
        estimator = LogisticRegression(C=0.3, max_iter=1500, random_state=42)
    else:
        estimator = HistGradientBoostingClassifier(max_iter=160, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=80, l2_regularization=8, early_stopping=False, random_state=42)
    return make_pipeline(preprocess, estimator)

def joint_prob(model, frame, temperature=1.0):
    raw = model.predict_proba(frame[FEATURES])
    # Explicitly map class numbers, even if an upstream estimator changes their order.
    p = np.full((len(frame),4), 1e-12)
    for j, label in enumerate(model.classes_):
        p[:, int(label)] = raw[:,j]
    return softmax(np.log(np.clip(p,1e-12,1))/temperature, axis=1)

def fit_calibrated(kind, fit_frame, calibration_frame):
    model = new_model(kind)
    model.fit(fit_frame[FEATURES], fit_frame.label.astype(int))
    logits = np.log(np.clip(joint_prob(model, calibration_frame),1e-12,1))
    y = calibration_frame.label.astype(int).to_numpy()
    optimum = minimize_scalar(lambda t: log_loss(y,softmax(logits/t,axis=1),labels=[0,1,2,3]), bounds=(0.4,3.0), method='bounded')
    if not optimum.success:
        raise RuntimeError('Calibration optimization failed.')
    return model, float(optimum.x)

def metrics(frame, p):
    y = frame.label.gt(0).astype(int)
    any_td = 1-p[:,0]
    return {'Brier':brier_score_loss(y,any_td), 'Log loss':log_loss(y,any_td,labels=[0,1]),
            'AUC':roc_auc_score(y,any_td), 'Mean predicted':any_td.mean(), 'Actual TD rate':y.mean(), 'N':len(y)}

# Include data values and code version in the key: corrections invalidate the fitted cache.
hash_cols = FEATURES + ['label','season','game_id','player_id']
fingerprint = hashlib.sha256(pd.util.hash_pandas_object(training[hash_cols],index=False).values.tobytes() + VERSION.encode()).hexdigest()[:18]
backtest_cache = CACHE / f'backtest_{fingerprint}.joblib'
if backtest_cache.exists() and not REBUILD_HISTORY:
    backtest = joblib.load(backtest_cache)
    print('Loaded cached backtest.')
else:
    candidates, records = {}, []
    for kind in ['Logistic','Boosted trees']:
        print('Fitting', kind, flush=True)
        model, temp = fit_calibrated(kind, fit_data, cal_data)
        candidates[kind] = (model,temp)
        records.append({'Model':kind, **metrics(valid_data,joint_prob(model,valid_data,temp))})
    validation_results = pd.DataFrame(records).sort_values('Brier')
    winner = validation_results.iloc[0].Model
    model, temp = candidates[winner]
    test_p = joint_prob(model,test_data,temp)
    # Baseline rates are estimated without using the validation or test outcomes.
    base_rates = fit_data.assign(scored=fit_data.label.gt(0)).groupby('position').scored.mean()
    base_p = test_data.position.map(base_rates).fillna(fit_data.label.gt(0).mean()).to_numpy()
    base_joint = np.column_stack([1-base_p,base_p,np.zeros(len(base_p)),np.zeros(len(base_p))])
    test_results = pd.DataFrame([{'Model':winner,**metrics(test_data,test_p)}, {'Model':'Position-only baseline',**metrics(test_data,base_joint)}])
    backtest = {'winner':winner, 'validation':validation_results, 'test_results':test_results, 'test_p':test_p}
    joblib.dump(backtest,backtest_cache)
winner = backtest['winner']
print(f'Validation season: {VALID_YEAR}. Selected: {winner}.')
display(backtest['validation'].round(4))
print(f'Untouched test season: {TEST_YEAR}.')
display(backtest['test_results'].round(4))
if backtest['test_results'].iloc[0].Brier >= backtest['test_results'].iloc[1].Brier:
    print('CAUTION: the selected model did not beat the position-only baseline on test Brier score.')
test_p = backtest['test_p']
calibration = pd.DataFrame({'probability':1-test_p[:,0], 'scored':test_data.label.gt(0).astype(int).to_numpy()})
calibration['bin'] = pd.cut(calibration.probability, np.linspace(0,1,11), include_lowest=True)
calibration_table = calibration.groupby('bin',observed=True).agg(predicted=('probability','mean'), actual=('scored','mean'), players=('scored','size')).reset_index()
display(calibration_table)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
ax.plot(calibration_table.predicted,calibration_table.actual,'o-',color='#087f8c',label=f'{TEST_YEAR} test')
ax.set(xlabel='Predicted chance of at least one TD', ylabel='Observed scoring rate', title='Does the predicted probability match the scoring rate?', xlim=(0,1), ylim=(0,1))
ax.legend(); plt.show()
print('Small bins are noisy; read the player counts alongside the chart.')
# Check component probabilities too; combined-TD accuracy does not prove each component is calibrated.
component_scores = []
for label, y, p in [('Rushing',test_data.label.isin([1,3]),test_p[:,1]+test_p[:,3]), ('Receiving',test_data.label.isin([2,3]),test_p[:,2]+test_p[:,3])]:
    component_scores.append({'Outcome':label,'Brier':brier_score_loss(y,p),'Mean predicted':p.mean(),'Actual rate':y.mean()})
display(pd.DataFrame(component_scores).round(4))

## Candidate-model comparisons
Both candidates use chronological splits and fixed checks. The injury comparison may be older because newer archives lack timestamps; that blocks release rather than inventing historical information.

In [ ]:
# A separate retrospective comparison; the original baseline backtest remains above.
# No 2025 report timestamp is invented. Data without verified timestamps has ctx_coverage=0.
def model_with_features(kind, numeric):
    preprocess=ColumnTransformer([
        ('numbers',make_pipeline(SimpleImputer(strategy='median',add_indicator=True),StandardScaler()),numeric),
        ('position',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['position'])])
    estimator=(LogisticRegression(C=.3,max_iter=1500,random_state=42) if kind=='Logistic' else
        HistGradientBoostingClassifier(max_iter=160,learning_rate=.05,max_leaf_nodes=15,min_samples_leaf=80,l2_regularization=8,early_stopping=False,random_state=42))
    return make_pipeline(preprocess,estimator)

def predict_feature_model(model, frame, numeric, temperature=1.):
    raw=model.predict_proba(frame[numeric+['position']])
    result=np.full((len(frame),4),1e-12)
    for j,label in enumerate(model.classes_): result[:,int(label)]=raw[:,j]
    return softmax(np.log(np.clip(result,1e-12,1))/temperature,axis=1)

def fit_feature_model(kind, fit_frame, cal_frame, numeric):
    model=model_with_features(kind,numeric)
    model.fit(fit_frame[numeric+['position']],fit_frame.label.astype(int))
    logits=np.log(np.clip(predict_feature_model(model,cal_frame,numeric),1e-12,1))
    opt=minimize_scalar(lambda t:log_loss(cal_frame.label.astype(int),softmax(logits/t,axis=1),labels=[0,1,2,3]),bounds=(.4,3.),method='bounded')
    if not opt.success:raise RuntimeError('Context calibration failed')
    return model,float(opt.x)

def paired_game_brier(frame, base, challenger, n_boot=500):
    y=frame.label.gt(0).astype(float).to_numpy()
    diff=(challenger-y)**2-(base-y)**2
    grouped=pd.DataFrame({'game':frame.game_id.to_numpy(),'difference':diff}).groupby('game').difference.agg(['sum','count'])
    if len(grouped)<2:return np.nan,np.nan
    rng=np.random.default_rng(42)
    indices=rng.integers(0,len(grouped),size=(n_boot,len(grouped)))
    values=grouped['sum'].to_numpy()[indices].sum(axis=1)/grouped['count'].to_numpy()[indices].sum(axis=1)
    return tuple(np.quantile(values,[.025,.975]))

comparison_records=[]
context_model_approved=False
context_approval_reason='Insufficient timestamped historical injury coverage.'
context_test_year=None
# Choose the last contiguous block with broad timestamped coverage, not the best scoring years.
covered_years=sorted(context_coverage.index[context_coverage.covered_fraction.ge(.60)].astype(int))
blocks=[y for y in covered_years if all(z in covered_years for z in [y-1,y-2]) and y-3>=FIRST_SEASON]
context_models={}
context_numeric=NUMERIC+CONTEXT_FEATURES
context_fp=hashlib.sha256(pd.util.hash_pandas_object(training[FEATURES+CONTEXT_FEATURES+['label','season','game_id','player_id']],index=False).values.tobytes()+INJURY_FEATURE_VERSION.encode()).hexdigest()[:18]
if blocks:
    context_approval_reason='Timestamped seasons exist, but chronological splits lack enough rows or outcome classes.'
    context_test_year=max(blocks)
    comparison_cal_year=context_test_year-2
    comparison_valid_year=context_test_year-1
    comparison_fit=training.loc[training.season.lt(comparison_cal_year)]
    comparison_cal=training.loc[training.season.eq(comparison_cal_year)]
    comparison_valid=training.loc[training.season.eq(comparison_valid_year)]
    comparison_test=training.loc[training.season.eq(context_test_year)]
    adequate=all(len(f)>=1000 and f.label.nunique()==4 for f in [comparison_fit,comparison_cal,comparison_valid,comparison_test])
    if adequate:
        # Fixed logistic architecture avoids choosing an architecture using the comparison test year.
        # Same fit/calibration/validation/test rows for both candidates.
        path=CACHE/f'context_compare_{context_fp}_{context_test_year}.joblib'
        if path.exists() and not REBUILD_HISTORY:
            bundle=joblib.load(path)
        else:
            bundle={}
            for label,numeric in [('Baseline',NUMERIC),('Injury context',context_numeric)]:
                print('Fitting comparison:',label,flush=True)
                model,temp=fit_feature_model('Logistic',comparison_fit,comparison_cal,numeric)
                bundle[label]={'model':model,'temperature':temp,
                    'valid':predict_feature_model(model,comparison_valid,numeric,temp),
                    'test':predict_feature_model(model,comparison_test,numeric,temp)}
            joblib.dump(bundle,path)
        context_models=bundle
        for split,frame in [('valid',comparison_valid),('test',comparison_test)]:
            for label in bundle:
                pp=bundle[label][split]
                for subgroup,mask in [('All',np.ones(len(frame),dtype=bool)),('Material offensive absence',frame.ctx_material_absence.to_numpy(dtype=bool))]:
                    if mask.sum()<2 or frame.loc[mask,'label'].gt(0).nunique()<2:continue
                    comparison_records.append({'split':split,'season':int(frame.season.iloc[0]),'model':label,'subgroup':subgroup,**metrics(frame.loc[mask],pp[mask]),'games':frame.loc[mask,'game_id'].nunique()})
        valid_base=metrics(comparison_valid,bundle['Baseline']['valid'])['Brier']
        valid_context=metrics(comparison_valid,bundle['Injury context']['valid'])['Brier']
        test_base=metrics(comparison_test,bundle['Baseline']['test'])['Brier']
        test_context=metrics(comparison_test,bundle['Injury context']['test'])['Brier']
        ci_low,ci_high=paired_game_brier(comparison_test,1-bundle['Baseline']['test'][:,0],1-bundle['Injury context']['test'][:,0])
        material=comparison_test.ctx_material_absence.to_numpy(dtype=bool)
        material_games=comparison_test.loc[material,'game_id'].nunique()
        subgroup_ok=False
        if material.sum()>=200 and material_games>=50:
            yy=comparison_test.loc[material,'label'].gt(0).astype(float).to_numpy()
            subgroup_ok=np.mean((1-bundle['Injury context']['test'][material,0]-yy)**2)<=np.mean((1-bundle['Baseline']['test'][material,0]-yy)**2)
        # Older results alone are not enough to release a new live injury model.
        recent_test=context_test_year>=SEASON-1
        context_model_approved=bool(valid_context<valid_base and test_context<test_base and ci_high<0 and subgroup_ok and recent_test)
        failures=[]
        if not recent_test:failures.append(f'newest timestamped test is {context_test_year}; need {SEASON-1} or later')
        if valid_context>=valid_base:failures.append('no validation Brier improvement')
        if test_context>=test_base or not ci_high<0:failures.append('held-out gain not clear at game-cluster level')
        if not subgroup_ok:failures.append('injury subgroup insufficient or worse')
        context_approval_reason='Approved by predefined checks' if context_model_approved else '; '.join(failures)
        print(f'Paired test Brier difference (injury minus baseline), 95% game-bootstrap interval: [{ci_low:.5f}, {ci_high:.5f}]')
comparison_results=pd.DataFrame(comparison_records)
if len(comparison_results):display(comparison_results.round(5))
print('Injury model eligible for automatic use:',context_model_approved)
print(context_approval_reason)
print('This is a retrospective participant-only benchmark. Weekly roster publication timestamps are not archived, and revised stats may differ from original releases. It is not a betting ROI test.')

# Fit an explicitly experimental current-context model for side-by-side inspection.
context_production_model=None
context_production_temperature=None
if context_models:
    weeks=training[['season','week']].drop_duplicates().sort_values(['season','week'])
    cal_keys=set(map(tuple,weeks.tail(8).to_numpy()))
    mask=pd.Series([(s,w) in cal_keys for s,w in zip(training.season,training.week)],index=training.index)
    # Calibration needs known context. Use latest covered weeks if the latest eight lack it.
    covered=training.loc[training.ctx_coverage.eq(1)]
    covered_keys=set(map(tuple,covered[['season','week']].drop_duplicates().sort_values(['season','week']).tail(8).to_numpy()))
    mask=pd.Series([(s,w) in covered_keys for s,w in zip(training.season,training.week)],index=training.index)
    cal=training.loc[mask & training.ctx_coverage.eq(1)]
    # Keep temporal order: never calibrate on old weeks after fitting later outcomes.
    if len(cal)>=500 and cal.label.nunique()==4:
        first_cal=cal.gameday.min()
        fit=training.loc[training.gameday.lt(first_cal)]
        if len(fit)>=2000 and fit.label.nunique()==4:
            path=CACHE/f'context_production_{context_fp}.joblib'
            if path.exists() and not REBUILD_HISTORY:
                context_production_model,context_production_temperature=joblib.load(path)
            else:
                context_production_model,context_production_temperature=fit_feature_model('Logistic',fit,cal,context_numeric)
                joblib.dump((context_production_model,context_production_temperature),path)
            print('Experimental injury model training cutoff:',first_cal.date(),'; calibration ends:',cal.gameday.max().date())


In [ ]:
# A second improvement can be tested on 2025 even when historical injury timestamps are missing.
# Last-appearance usage helps detect a role change sooner than a four-game average.
RECENCY_FEATURES=[]
combined=pd.concat([training.assign(_source='train',_row=training.index),prediction_rows.assign(_source='live',_row=prediction_rows.index)],ignore_index=True).sort_values(['player_id','gameday','game_id'])
for metric in ['carries','targets','offense_pct','rush_inside5','rz_targets']:
    col=metric+'_previous_appearance'
    combined[col]=combined.groupby('player_id')[metric].shift(1)
    delta=metric+'_recent_change'
    combined[delta]=combined[col]-combined[metric+'_last4']
    RECENCY_FEATURES.extend([col,delta])
for source,frame in [('train',training),('live',prediction_rows)]:
    lookup=combined.loc[combined._source.eq(source)].set_index('_row')
    for col in RECENCY_FEATURES:frame[col]=lookup[col].reindex(frame.index).to_numpy()
recency_numeric=NUMERIC+RECENCY_FEATURES
recency_fp=hashlib.sha256(pd.util.hash_pandas_object(training[recency_numeric+['position','label','season','game_id','player_id']],index=False).values.tobytes()+b'recency-v1').hexdigest()[:18]
recency_path=CACHE/f'recency_compare_{recency_fp}_{winner}.joblib'
if recency_path.exists() and not REBUILD_HISTORY:
    recency_bundle=joblib.load(recency_path)
else:
    rfit=training.loc[training.season.lt(CAL_YEAR)]
    rcal=training.loc[training.season.eq(CAL_YEAR)]
    rm,rt=fit_feature_model(winner,rfit,rcal,recency_numeric)
    recency_bundle={'valid':predict_feature_model(rm,training.loc[training.season.eq(VALID_YEAR)],recency_numeric,rt),
                    'test':predict_feature_model(rm,training.loc[training.season.eq(TEST_YEAR)],recency_numeric,rt)}
    joblib.dump(recency_bundle,recency_path)
rvalid=training.loc[training.season.eq(VALID_YEAR)]
rtest=training.loc[training.season.eq(TEST_YEAR)]
recency_valid_score=metrics(rvalid,recency_bundle['valid'])['Brier']
baseline_valid_score=float(backtest['validation'].loc[backtest['validation'].Model.eq(winner),'Brier'].iloc[0])
recency_test_score=metrics(rtest,recency_bundle['test'])['Brier']
baseline_test_score=metrics(rtest,backtest['test_p'])['Brier']
recency_ci=paired_game_brier(rtest,1-backtest['test_p'][:,0],1-recency_bundle['test'][:,0])
recency_approved=bool(recency_valid_score<baseline_valid_score and recency_test_score<baseline_test_score and recency_ci[1]<0)
recency_results=pd.DataFrame([
    {'model':'Original baseline','validation_Brier':baseline_valid_score,'test_Brier':baseline_test_score},
    {'model':'Recent usage','validation_Brier':recency_valid_score,'test_Brier':recency_test_score}])
print(f'Recent usage comparison: validation {VALID_YEAR}, later test {TEST_YEAR}')
display(recency_results.round(5))
print('95% game-bootstrap Brier difference interval:',recency_ci)
print('Recent usage model eligible for use:',recency_approved)
recency_production_model=None
recency_production_temperature=None
if recency_approved:
    weeks=training[['season','week']].drop_duplicates().sort_values(['season','week'])
    keys=set(map(tuple,weeks.tail(8).to_numpy()))
    recent=pd.Series([(s,w) in keys for s,w in zip(training.season,training.week)],index=training.index)
    rp=CACHE/f'recency_production_{recency_fp}_{winner}.joblib'
    if rp.exists() and not REBUILD_HISTORY:
        recency_production_model,recency_production_temperature=joblib.load(rp)
    else:
        recency_production_model,recency_production_temperature=fit_feature_model(winner,training.loc[~recent],training.loc[recent],recency_numeric)
        joblib.dump((recency_production_model,recency_production_temperature),rp)


## 6. Fit the chosen live model and rank the slate

The original model is always available. Recent-usage predictions replace it only after the historical comparison passes. Injury-context predictions are shown separately as experimental unless their release criteria pass; they cannot quietly become betting recommendations. Expected QB identity is inferred from prior participation, roster membership and injury status, not confirmed by a depth chart.

The chosen probability and model label are saved. Injury coverage and meaningful role uncertainties still gate the review list. `role_adjusted=False` means no manual/empirical workload scenario, not that recent usage and injuries were never checked.


In [ ]:
week_keys = training[['season','week']].drop_duplicates().sort_values(['season','week'])
cal_weeks = set(map(tuple,week_keys.tail(8).to_numpy()))
is_recent = pd.Series([(s,w) in cal_weeks for s,w in zip(training.season,training.week)],index=training.index)
production_fit = training.loc[~is_recent]
production_cal = training.loc[is_recent]
model_cache = CACHE / f'production_{fingerprint}_{winner.replace(" ","_")}.joblib'
if model_cache.exists() and not REBUILD_HISTORY:
    production_model, production_temperature = joblib.load(model_cache)
    print('Loaded cached current model.')
else:
    print('Fitting current model...', flush=True)
    production_model, production_temperature = fit_calibrated(winner,production_fit,production_cal)
    joblib.dump((production_model,production_temperature),model_cache)
baseline_rows = prediction_rows.copy()
for col in baseline_rows.columns:
    if col.startswith('baseline_') and col[9:] in FEATURES:
        baseline_rows[col[9:]] = baseline_rows[col]
baseline_p = joint_prob(production_model,baseline_rows,production_temperature)
p = joint_prob(production_model,prediction_rows,production_temperature)
live_model_labels=np.full(len(prediction_rows),'original_baseline_v1',dtype=object)
experimental_injury_probability=np.full(len(prediction_rows),np.nan)
if recency_approved and recency_production_model is not None:
    p=predict_feature_model(recency_production_model,prediction_rows,recency_numeric,recency_production_temperature)
    live_model_labels[:]='recent_usage_v1'
if context_production_model is not None:
    context_p=predict_feature_model(context_production_model,prediction_rows,context_numeric,context_production_temperature)
    experimental_injury_probability=1-context_p[:,0]
    covered=prediction_rows.ctx_coverage.eq(1).to_numpy()
    experimental_injury_probability[~covered]=np.nan
    # Never combine independently tested changes without a head-to-head test.
    use_context=(context_model_approved and not recency_approved)
    if use_context:
        safe=(prediction_rows.ctx_coverage.eq(1) & prediction_rows.ctx_qb_known &
              prediction_rows.ctx_reason.eq('PREGAME_CONTEXT_AVAILABLE') &
              prediction_rows.injury_coverage_ok & prediction_rows.new_team.eq(0) &
              prediction_rows.days_since_played.le(21) & prediction_rows.prior_games.ge(4) &
              prediction_rows.injury_status.isin(['ACTIVE','NOT_LISTED','NO_DESIGNATION','PROBABLE']) &
              prediction_rows.offense_pct_last4.fillna(0).ge(.15))
        if roster_week<WEEK:safe[:]=False
        # A projected backup QB is not made eligible just because he is healthy.
        safe &= ~prediction_rows.position.eq('QB') | prediction_rows.ctx_is_expected_qb.eq(1)
        p[safe.to_numpy()]=context_p[safe.to_numpy()]
        prediction_rows.loc[safe,'review_eligible']=True
        prediction_rows.loc[safe,'role_uncertain']=False
        live_model_labels[safe.to_numpy()]='injury_context_v1'
# Team-level context is retained for diagnostics, but does not blanket-block
# unrelated positions. apply_injuries_and_roles handles direct teammate exposure.
# Check qb_known separately: ctx_reason may prioritize a teammate over an unknown QB.
unresolved_context=(prediction_rows.ctx_coverage.eq(1) & ~prediction_rows.ctx_qb_known)
prediction_rows.loc[unresolved_context,'review_eligible']=False
prediction_rows.loc[unresolved_context,'role_uncertain']=True


# Apply this AFTER every eligibility override so low-volume players cannot re-enter.
prediction_rows['opportunity_ok'] = prediction_rows.apply(opportunity_screen, axis=1)
prediction_rows['review_eligible'] &= prediction_rows.opportunity_ok

def review_reasons(row):
    reasons = []
    if not row.injury_coverage_ok: reasons.append('injury coverage unknown')
    if row.injury_status not in {'ACTIVE','NOT_LISTED','NO_DESIGNATION','PROBABLE'}: reasons.append('own availability unresolved')
    if row.role_uncertain: reasons.append('direct role / QB / return uncertainty')
    if row.new_team: reasons.append('new team')
    if row.prior_games < 4: reasons.append('limited history')
    if pd.isna(row.offense_pct_last4) or row.offense_pct_last4 < .15: reasons.append('low/unknown snap role')
    if not row.opportunity_ok: reasons.append('low/unknown scoring opportunity')
    if roster_week < WEEK: reasons.append('older roster snapshot')
    if row.role_adjusted: reasons.append('unvalidated workload scenario')
    return '; '.join(reasons) if reasons else 'passes screening; final active status unconfirmed'
prediction_rows['review_reason'] = prediction_rows.apply(review_reasons, axis=1)
# Refresh the audit after all context/model eligibility changes; excluded rows remain.
for column in ['role_uncertain','review_eligible','opportunity_ok','review_reason']:
    values = prediction_rows.set_index(['game_id','player_id'])[column]
    keys = pd.MultiIndex.from_frame(availability_audit[['game_id','player_id']])
    mapped = values.reindex(keys)
    keep = keys.isin(values.index)
    if column not in availability_audit: availability_audit[column] = None
    availability_audit.loc[keep,column] = mapped.to_numpy()[keep]

rankings = prediction_rows[['player_id','player','position','team','opponent','game_id','gameday','gametime','prior_games','new_team','role_adjusted']].copy().reset_index(drop=True)
rankings['rush_td_probability'] = p[:,1]+p[:,3]
rankings['receiving_td_probability'] = p[:,2]+p[:,3]
rankings['any_td_probability'] = 1-p[:,0]
rankings['both_types_probability'] = p[:,3]
for col in ['carries_last4','targets_last4','rush_inside5_last4','rz_targets_last4','offense_pct_last4']:
    rankings[col] = prediction_rows[col].to_numpy()
rankings['baseline_any_td_probability'] = 1-baseline_p[:,0]
rankings['model_label'] = live_model_labels
rankings['experimental_injury_probability'] = experimental_injury_probability
rankings['injury_model_approved'] = context_model_approved
rankings['injury_approval_reason'] = context_approval_reason
for col in ['ctx_expected_qb_id','ctx_qb_known','ctx_reason','ctx_coverage']:
    rankings[col] = prediction_rows[col].to_numpy()
rankings['model_delta_pp'] = 100*(rankings.any_td_probability-rankings.baseline_any_td_probability)
for col in ['injury_status','injury_source','injury_timestamp_basis','injury_checked_at_utc','injury_report_updated_at_utc',
            'injury_coverage_ok','role_adjustment_source','role_adjustment_games','role_context',
            'role_uncertain','review_eligible','direct_role_change','opportunity_ok','review_reason']:
    rankings[col] = prediction_rows[col].to_numpy()
for col in prediction_rows.columns:
    if col.startswith('baseline_'):
        rankings[col] = prediction_rows[col].to_numpy()
rankings['flags'] = 'INJURY STATUS: ' + rankings.injury_status + ' | CHECK FINAL ACTIVE LIST'
rankings.loc[~rankings.injury_coverage_ok,'flags'] += ' | INJURY COVERAGE UNKNOWN'
rankings.loc[rankings.role_uncertain,'flags'] += ' | ROLE / TEAM CONTEXT UNCERTAIN'
rankings.loc[~rankings.opportunity_ok,'flags'] += ' | LOW/UNKNOWN SCORING OPPORTUNITY'
rankings.loc[~rankings.review_eligible,'flags'] += ' | NOT A REVIEW CANDIDATE'
for mask, message in [
    (rankings.prior_games.lt(4),' | LIMITED HISTORY'),
    (rankings.new_team.eq(1),' | NEW TEAM'),
    (rankings.offense_pct_last4.fillna(0).lt(0.15),' | LOW/UNKNOWN ROLE'),
    (rankings.role_adjusted,' | UNVALIDATED WORKLOAD SCENARIO')]:
    rankings.loc[mask,'flags'] += message
if roster_week < WEEK:
    rankings['flags'] += ' | OLDER ROSTER SNAPSHOT'
rankings['roster_week'] = roster_week
rankings['stats_cutoff_exclusive'] = str(week_start.date())
rankings['generated_at_et'] = pd.Timestamp.now(tz='America/New_York').isoformat()
rankings = rankings.sort_values('any_td_probability',ascending=False).reset_index(drop=True)
rankings.insert(0,'rank',np.arange(1,len(rankings)+1))
assert np.allclose(p.sum(axis=1),1)
assert (rankings.any_td_probability >= rankings.rush_td_probability-1e-10).all()
assert (rankings.any_td_probability >= rankings.receiving_td_probability-1e-10).all()
assert np.allclose(rankings.any_td_probability, rankings.rush_td_probability+rankings.receiving_td_probability-rankings.both_types_probability)
shown = rankings[['rank','player','position','team','opponent','any_td_probability','rush_td_probability','receiving_td_probability','carries_last4','targets_last4','rush_inside5_last4','injury_status','role_adjustment_source','flags']].head(TOP_N)
display(shown.style.format({'any_td_probability':'{:.1%}','rush_td_probability':'{:.1%}','receiving_td_probability':'{:.1%}','carries_last4':'{:.1f}','targets_last4':'{:.1f}','rush_inside5_last4':'{:.2f}'},na_rep='No history').background_gradient(subset=['any_td_probability'],cmap='YlGnBu'))
print('The CSV includes every player, not just the top rows displayed above.')
print('FINAL SCREENING SUMMARY (reasons can overlap):')
print(f'{int(rankings.review_eligible.sum())} of {len(rankings)} players pass review screening.')
display(rankings.review_reason.str.split('; ').explode().value_counts().rename_axis('reason').to_frame('players'))
print('Injury model approval:', context_model_approved, '|', context_approval_reason)
print('Screening v2 uses heuristic direct-role and opportunity rules; predictive improvement is not established.')


## 7. Save the results

The rankings CSV stores probabilities as decimals: **0.35 means 35%**. The backtest and calibration CSVs document model performance. The run summary records the settings and cutoff. These exports use ordinary CSV files so they can be opened in Excel, Sheets, or Python.

To update another day in the same Colab session: change **Settings**, then rerun from Settings to the end. Cached historical data and unchanged models are reused. For a fresh Colab session, run all cells again; enable Drive caching if you want to reuse the previous session's downloads and models.

In [ ]:
tag = f'{target.date()}_to_{end.date()}' if target != end else str(target.date())
# Keep different same-day slates from overwriting one another.
slate_key = hashlib.sha256('|'.join(sorted(selected.game_id)).encode()).hexdigest()[:6]
stem = f'NFL_TD_{tag}_{slate_key}'
csv_path = OUTPUT / f'{stem}_Rankings.csv'
rankings.to_csv(csv_path,index=False)
backtest['test_results'].to_csv(OUTPUT / f'{stem}_Backtest.csv',index=False)
calibration_table.to_csv(OUTPUT / f'{stem}_Calibration.csv',index=False)
manifest = {'model_version':VERSION,'selected_model':winner,'season':SEASON,'week':WEEK,
            'games':selected.game_id.tolist(),'generated_at_et':NOW.isoformat(),
            'stats_cutoff_exclusive':str(week_start.date()),'roster_week':roster_week,
            'excluded_players':EXCLUDE_PLAYERS,'role_overrides':ROLE_OVERRIDES,'extra_players':EXTRA_PLAYERS,
            'conditional_on':'at least one offensive snap','test_season':TEST_YEAR,
            'production_temperature':production_temperature,'data_fingerprint':fingerprint}
(OUTPUT / f'{stem}_Run_Info.json').write_text(json.dumps(manifest,indent=2))
print('Rankings saved:',csv_path)
print('Backtest, calibration and run settings saved in:',OUTPUT)
availability_audit.to_csv(OUTPUT / f'{stem}_Availability_Audit.csv', index=False)
injury_report.to_csv(OUTPUT / f'{stem}_Injury_Source.csv', index=False)
manifest.update(injury_layer_version='injury-v2.0', injury_source=injury_feed_source,
                injury_max_age_hours=INJURY_MAX_AGE_HOURS,
                availability_overrides=AVAILABILITY_OVERRIDES,
                auto_role_scenarios=AUTO_ROLE_SCENARIOS,
                role_scenarios_backtested=False,
                eligible_review_players=int(rankings.review_eligible.sum()))
(OUTPUT / f'{stem}_Run_Info.json').write_text(json.dumps(manifest,indent=2))
print('Injury audit and report snapshot saved. Review-eligible players:', rankings.review_eligible.sum())

comparison_results.to_csv(OUTPUT / f'{stem}_Injury_Comparison.csv',index=False)
recency_results.to_csv(OUTPUT / f'{stem}_Recent_Usage_Comparison.csv',index=False)
archive_status.to_csv(OUTPUT / f'{stem}_Injury_Archive_Coverage.csv',index=False)
manifest.update(notebook_version='sunday-screening-v2',recency_model_approved=recency_approved,
    context_model_approved=context_model_approved,context_approval_reason=context_approval_reason,
    screening_rules={'rb_opportunities':MIN_REVIEW_RB_OPPORTUNITIES, 'receiver_targets':MIN_REVIEW_RECEIVER_TARGETS, 'rz_opportunities':MIN_REVIEW_RZ_OPPORTUNITIES, 'qb_carries':MIN_REVIEW_QB_CARRIES, 'qb_inside5':MIN_REVIEW_QB_INSIDE5},
    context_test_year=context_test_year,recency_fingerprint=recency_fp,context_fingerprint=context_fp,
    selected_model_labels=sorted(rankings.model_label.unique().tolist()),generated_at_et=rankings.generated_at_et.iloc[0])
(OUTPUT / f'{stem}_Run_Info.json').write_text(json.dumps(manifest,indent=2))


In [ ]:
# OPTIONAL: download the rankings directly to your computer.
# Uncomment these two lines in Colab if you want the browser download.
from google.colab import files
files.download(str(csv_path))

## Reading the results

- `any_td_probability`: chosen model's chance of at least one rushing/receiving TD, conditional on an offensive snap.
- `model_label`: original baseline, recent-usage model, or approved injury model.
- `baseline_any_td_probability`: original baseline for comparison; `model_delta_pp` is the chosen model's difference.
- `experimental_injury_probability`: a separate research projection, **not** the betting probability unless the injury model passed release criteria and was selected.
- `review_eligible`: passes availability/role checks, not necessarily price/value checks.
- `ctx_expected_qb_id`: inferred QB based on prior usage and available roster/injury reports; not a confirmed starter.
- `flags`: explain exclusions. No-history reserves do not automatically trigger whole-team flags.

The model still lacks a complete treatment of weather, offensive-line quality, defensive injuries, routes, and game-day coaching decisions. Automated checks do not guarantee that a player is active. Review final inactive reports before betting.

Historical injury features only accept report timestamps before kickoff, within 96 hours. A report's timestamp is not proof of a complete original archive; roster publication times and revision histories are not fully available. Untimestamped newer files cannot be backfilled using today's timestamps. Live file publication timestamps are permitted only for upcoming games.

Model performance is evaluated with Brier scores (lower is better), calibration, and paired resampling by game. These are accuracy checks, not profit estimates. The EV buffer is a fixed screening assumption, not a calibrated uncertainty interval. Parlays are not priced: multiplying player probabilities assumes independence.

Sources: [nflverse](https://github.com/nflverse/nflverse-data), [injury dictionary](https://nflreadr.nflverse.com/articles/dictionary_injuries.html), [data schedule](https://nflreadr.nflverse.com/articles/nflverse_data_schedule.html).


## 8. Automatic TD odds and value — daily budget mode
Use the same `ODDS_API_KEY` Colab Secret as NHL and enable access for this notebook. Google Drive caching is now on by default. Keep `END_DATE` blank to fetch only the selected day: Thursday games on Thursday, Sunday games on Sunday, Monday games on Monday. Run **before kickoff**. The existing blank-date setting selects the next unstarted game day; check the displayed slate.

Only `player_anytime_td` is requested, from four specified books: up to one credit per matched game. Available players in a game are returned together. Model training uses no odds credits. The cache saves successful and empty responses for the New York calendar day, so reruns do not refresh odds. New days or new games fetch again.

For a deliberate same-day refresh, change `ODDS_REFRESH_TOKEN` in Settings to a **new** label, rerun Settings and the odds cells. The same label only refreshes once per game that day, even after a runtime restart. Clear it afterward. Empty/missing markets never trigger automatic retries or alternate-market calls.

A free live quota check tracks the account total across both models. New paid calls stop at the 25-credit reserve or 20-credit run limit. This does not allocate separate monthly sport budgets. Keep Drive enabled to preserve saved odds between sessions.

**Market scope matters:** the model covers rushing OR receiving TDs only, conditional on an offensive snap. Anytime-TD book rules can also count returns or other TD types; passing TDs normally are not scorer TDs. The displayed EV uses only the model's narrower scoring probability, not a calibrated forecast of all eligible market outcomes. All rows flag this mismatch; review settlement and availability. No extra TD probability is invented.

Saved snapshots retain original timestamps; prices older than one hour are excluded from the current value board. Underdog candidates are separate, with quote age and any provider multiplier. Indicative DFS prices are not used for straight-bet EV or a parlay payout. Verify the actual slip.

The 3-percentage-point buffer is a sensitivity check, not a confidence interval. This is not a historical betting-profitability backtest. References: [API/quota rules](https://the-odds-api.com/liveapi/guides/v4/), [markets](https://the-odds-api.com/sports-odds-data/betting-markets.html).

The displayed value board is diagnostic. Only the separate review-candidate table applies the injury/role gate, and that is still not proof of a profitable bet. Underdog listings include the same eligibility and uncertainty flags.


In [ ]:
import unicodedata, re
ODDS_BOOKMAKERS = ["fanduel", "draftkings", "betmgm", "underdog"]
ODDS_MARKETS = ["player_anytime_td"]
PLAYER_NAME_ALIASES = {}  # Explicit provider-name -> exact model-name mappings only.
MAX_ODDS_AGE_HOURS = 1.0
MAX_PREDICTION_AGE_HOURS = 24.0
MIN_ESTIMATED_EV_PCT = 3.0
MIN_BUFFERED_EV_PCT = 5.0  # Conservative screening rule, NOT an optimized/validated profit threshold.
PROBABILITY_BUFFER_PP = 3.0
MIN_PRIOR_GAMES = 4
# Shared behavior in both notebooks; no API key is written to disk.
import os, json, hashlib, math, sys, time, urllib.request, urllib.parse, urllib.error
from pathlib import Path
import pandas as pd

class OddsFeedError(RuntimeError):
    pass

class BudgetOddsClient:
    def __init__(self, root, api_key, refresh_token='', reserve=25, run_limit=20):
        self.api_key = api_key
        self.root = Path(root) / 'odds_daily_cache'
        self.root.mkdir(parents=True, exist_ok=True)
        self.refresh_token = str(refresh_token).strip()
        self.reserve, self.run_limit = int(reserve), int(run_limit)
        if self.reserve < 0 or self.run_limit < 0:
            raise OddsFeedError('Credit limits cannot be negative.')
        self.remaining = None
        self.spent = 0
        self.account = hashlib.sha256(api_key.encode()).hexdigest()[:16]

    def today(self):
        return str(pd.Timestamp.now(tz='America/New_York').date())

    def _network(self, path, params):
        # No automatic retries: an interrupted response might already have cost credits.
        query = urllib.parse.urlencode(dict(params, apiKey=self.api_key))
        req = urllib.request.Request('https://api.the-odds-api.com/v4/' + path + '?' + query,
                                     headers={'Accept':'application/json'})
        try:
            with urllib.request.urlopen(req, timeout=35) as response:
                raw = response.read()
                headers = dict(response.headers)
            data = json.loads(raw)
        except urllib.error.HTTPError as err:
            raise OddsFeedError(f'Odds request stopped (HTTP {err.code}). Check key, coverage or quota; no automatic retry.') from None
        except (urllib.error.URLError, TimeoutError, OSError, ValueError):
            raise OddsFeedError('Odds response unavailable. No automatic retry; check connection and account usage before retrying.') from None
        if not isinstance(data, (dict,list)) or (isinstance(data,dict) and 'error_code' in data):
            raise OddsFeedError('Unexpected odds response; check the provider dashboard.')
        return data, {str(k).lower():v for k,v in headers.items()}

    def check_quota(self):
        # /sports is free; live quota covers both notebooks and other uses of this key.
        _, headers = self._network('sports', {})
        try:
            self.remaining = int(float(headers['x-requests-remaining']))
        except (KeyError, TypeError, ValueError):
            raise OddsFeedError('Could not verify remaining credits; paid requests disabled.') from None
        print(f'Account credits remaining: {self.remaining} | Protected reserve: {self.reserve}')

    def request(self, path, params):
        chargeable = path.endswith('/odds')
        if not chargeable:
            data, _ = self._network(path, params)  # Event discovery is free and stays current.
            return data
        markets = set(filter(None, str(params.get('markets','')).split(',')))
        books = set(filter(None, str(params.get('bookmakers','')).split(',')))
        if len(markets) != 1 or not 1 <= len(books) <= 10:
            raise OddsFeedError('Budget mode requires exactly one market and 1–10 explicit bookmakers.')
        cache_key = hashlib.sha256(json.dumps([self.account,path,params],sort_keys=True).encode()).hexdigest()
        cache_path = self.root / (cache_key + '.json')
        cached = None
        if cache_path.exists():
            try:
                cached = json.loads(cache_path.read_text())
            except (ValueError,OSError):
                raise OddsFeedError('Saved odds cache is unreadable. Inspect it before spending another request.') from None
            # An unchanged token never repeatedly refreshes, even across Colab restarts.
            if cached.get('fetch_day_et') == self.today() and (not self.refresh_token or cached.get('refresh_token') == self.refresh_token):
                print('SAVED ODDS:', cached.get('fetched_at_utc'), '| 0 credits; quote times remain unchanged.')
                return cached['payload']
        if self.remaining is None:
            self.check_quota()
        if self.remaining - 1 < self.reserve or self.spent + 1 > self.run_limit:
            raise OddsFeedError('Credit guard stopped new odds requests. Saved results remain available; predictions are unaffected.')
        # Reserve one credit before attempting, even if the response is interrupted.
        self.spent += 1
        self.remaining -= 1
        data, headers = self._network(path, params)
        try:
            self.remaining = int(float(headers['x-requests-remaining']))
        except (KeyError, TypeError, ValueError):
            self.remaining = None  # Force a free authoritative quota check before any next fetch.
        print(f'New odds fetch: cost {headers.get("x-requests-last", "up to 1")}; account remaining: {self.remaining}')
        record = {'fetch_day_et':self.today(), 'fetched_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),
                  'refresh_token':self.refresh_token, 'payload':data}
        tmp = cache_path.with_suffix('.tmp')
        tmp.write_text(json.dumps(record)); tmp.replace(cache_path)
        return data

def odds_api_key():
    key = os.environ.get('ODDS_API_KEY','').strip()
    if key:
        return key
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            return str(userdata.get('ODDS_API_KEY')).strip()
        except Exception:
            pass
    return ''

def odds_request(path, params, api_key):
    return _odds_client.request(path, params)

def american_to_decimal(odds):
    # American odds express net winnings relative to a 100-unit stake or payout.
    value = float(str(odds).strip().replace("−", "-"))
    if not np.isfinite(value) or abs(value) < 100:
        raise ValueError("American odds must be >= +100 or <= -100. Use +100 for even money.")
    return 1 + (value / 100 if value > 0 else 100 / abs(value))

def fair_american(probability):
    if probability >= 0.5:
        return -100 * probability / (1 - probability)
    return 100 * (1 - probability) / probability

def has_value(value):
    return value is not None and not pd.isna(value) and str(value).strip() != ""

def explicit_utc(value):
    if not has_value(value):
        raise ValueError("Missing odds timestamp.")
    result = pd.Timestamp(value)
    if pd.isna(result) or result.tzinfo is None:
        raise ValueError("Timestamp must include a timezone, e.g. Z or -04:00.")
    return result.tz_convert("UTC")

def confirmed_bool(value):
    # Do not treat the CSV string 'False' as True.
    return str(value).strip().casefold() in {"true", "yes", "1"}

def compare_odds(ranked, quotes, now=None):
    now = pd.Timestamp.now(tz="UTC") if now is None else explicit_utc(now)
    if not 0 <= PROBABILITY_BUFFER_PP < 100:
        raise ValueError("Probability buffer must be between 0 and 100 percentage points.")
    if MAX_ODDS_AGE_HOURS <= 0 or MAX_PREDICTION_AGE_HOURS <= 0:
        raise ValueError("Freshness limits must be positive.")
    accepted, rejected = [], []
    for row_number, row in enumerate(quotes.to_dict("records"), start=1):
        if not has_value(row.get("over_american")):
            continue                  # Unfilled template rows are not errors.
        try:
            if float(row.get("line", 0.5)) != 0.5:
                raise ValueError("Only the 1+ TD selection is supported.")
            book = str(row.get("book", "")).strip() if has_value(row.get("book")) else ""
            if not book:
                raise ValueError("Missing sportsbook name.")
            if book.casefold().replace(" ", "") in {"underdog", "underdogfantasy", "underdogsports"}:
                raise ValueError("Underdog DFS prices are indicative; use the separate Underdog candidate table.")
            matches = ranked.copy()
            if has_value(row.get("player_id")):
                matches = matches[matches.player_id.astype(str) == str(row["player_id"])]
            elif has_value(row.get("player")):
                matches = matches[matches.player.str.strip().str.casefold() == str(row["player"]).strip().casefold()]
            else:
                raise ValueError("Provide player_id or an exact player name.")
            if has_value(row.get("game_id")):
                matches = matches[matches.game_id.astype(str) == str(row["game_id"])]
            if len(matches) != 1:
                raise ValueError("Player/game does not match exactly one current ranking. Check name and IDs.")
            player = matches.iloc[0]
            # Guard against a copied price row whose name disagrees with its ID.
            if has_value(row.get("player")) and str(row["player"]).strip().casefold() != str(player.player).strip().casefold():
                raise ValueError("Player name does not match the supplied ID.")
            if has_value(row.get("team")) and str(row["team"]).upper().strip() != str(player.team).upper():
                raise ValueError("Team does not match the current ranking.")
            quoted = explicit_utc(row.get("quoted_at_utc"))
            quote_age = (now - quoted).total_seconds() / 3600
            if quote_age < -5 / 60:
                raise ValueError("Quote timestamp is in the future.")
            if quote_age > MAX_ODDS_AGE_HOURS:
                raise ValueError("Odds are stale; recheck the price and timestamp.")
            if explicit_utc(player.start_utc) <= now:
                raise ValueError("Game has already started.")
            prediction_age = (now - explicit_utc(player.generated_at_utc)).total_seconds() / 3600
            if prediction_age > MAX_PREDICTION_AGE_HOURS or prediction_age < -5 / 60:
                raise ValueError("Prediction timestamp is stale or invalid; rerun rankings.")
            probability = float(player.probability)
            if not np.isfinite(probability) or not 0 < probability < 1:
                raise ValueError("Invalid model probability.")
            decimal = american_to_decimal(row["over_american"])
            implied = 1 / decimal
            market_probability, overround = np.nan, np.nan
            under = row.get("under_american")
            if has_value(under):
                under_implied = 1 / american_to_decimal(under)
                market_probability = implied / (implied + under_implied)
                overround = 100 * (implied + under_implied - 1)
            buffered_probability = max(0, probability - PROBABILITY_BUFFER_PP / 100)
            ev_pct = 100 * (probability * decimal - 1)
            buffered_ev = 100 * (buffered_probability * decimal - 1)
            enough_history = int(player.prior_games) >= MIN_PRIOR_GAMES
            confirmed = confirmed_bool(row.get("playing_confirmed", False))
            flags = [str(player.get("flags", "")), "Rush/receiving model only; verify anytime-TD settlement scope"]
            if not confirmed:
                flags.append("Playing status unconfirmed")
            if not enough_history:
                flags.append("Limited NFL history")
            if has_value(player.get("layoff_flag")):
                flags.append(str(player.layoff_flag))
            if pd.notna(overround) and overround < 0:
                flags.append("Over/under pair implies negative margin; verify prices")
            accepted.append({
                "player": player.player, "player_id": str(player.player_id), "game_id": str(player.game_id),
                "team": player.team, "opponent": player.opponent, "book": book, "line": 0.5,
                "over_american": float(str(row["over_american"]).replace("−", "-")),
                "under_american": float(str(under).replace("−", "-")) if has_value(under) else np.nan,
                "decimal_odds": decimal, "model_chance_pct": 100 * probability,
                "break_even_pct": 100 * implied, "edge_pp": 100 * (probability - implied),
                "estimated_ev_pct": ev_pct, "estimated_net_per_100_staked": ev_pct,
                "buffered_ev_pct": buffered_ev, "fair_american": fair_american(probability),
                "market_no_vig_pct": 100 * market_probability,
                "no_vig_edge_pp": 100 * (probability - market_probability), "book_overround_pct": overround,
                "quote_age_minutes": max(0, quote_age * 60), "quoted_at_utc": quoted.isoformat(),
                "playing_confirmed": confirmed, "review_eligible": confirmed_bool(player.get("review_eligible", False)), "flags": "; ".join(flags),
                "passes_value_filters": bool(ev_pct >= MIN_ESTIMATED_EV_PCT and buffered_ev >= MIN_BUFFERED_EV_PCT and enough_history and confirmed_bool(player.get("review_eligible", False))),
                "prediction_generated_at_utc": str(player.generated_at_utc),
                "model_fingerprint": player.model_fingerprint,
                "source_market": row.get("source_market", ""),
                "model_scope": "Rushing or receiving TD, conditional on offensive participation",
                "market_scope": "Anytime TD: may include additional scoring types; verify book rules",
            })
        except (ValueError, TypeError, KeyError, OverflowError) as err:
            rejected.append({"input_row": row_number, "player": row.get("player", row.get("player_id", "")),
                             "book": row.get("book", ""), "reason": str(err)})
    board = pd.DataFrame(accepted)
    if len(board):
        # Multiple versions from one book: use its latest quote, not the most attractive old one.
        board = board.sort_values("quoted_at_utc").drop_duplicates(["game_id", "player_id", "book"], keep="last")
        board = board.sort_values("estimated_ev_pct", ascending=False).reset_index(drop=True)
        board.insert(0, "value_rank", np.arange(1, len(board) + 1))
    return board, pd.DataFrame(rejected)
def normalize_label(value):
    plain = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]", "", plain)

# Explicit team names and exact player aliases; no fuzzy matching.
TEAM_NAMES = {'Arizona Cardinals': 'ARI', 'Atlanta Falcons': 'ATL', 'Baltimore Ravens': 'BAL', 'Buffalo Bills': 'BUF', 'Carolina Panthers': 'CAR', 'Chicago Bears': 'CHI', 'Cincinnati Bengals': 'CIN', 'Cleveland Browns': 'CLE', 'Dallas Cowboys': 'DAL', 'Denver Broncos': 'DEN', 'Detroit Lions': 'DET', 'Green Bay Packers': 'GB', 'Houston Texans': 'HOU', 'Indianapolis Colts': 'IND', 'Jacksonville Jaguars': 'JAX', 'Kansas City Chiefs': 'KC', 'Las Vegas Raiders': 'LV', 'Los Angeles Chargers': 'LAC', 'Los Angeles Rams': 'LA', 'Miami Dolphins': 'MIA', 'Minnesota Vikings': 'MIN', 'New England Patriots': 'NE', 'New Orleans Saints': 'NO', 'New York Giants': 'NYG', 'New York Jets': 'NYJ', 'Philadelphia Eagles': 'PHI', 'Pittsburgh Steelers': 'PIT', 'San Francisco 49ers': 'SF', 'Seattle Seahawks': 'SEA', 'Tampa Bay Buccaneers': 'TB', 'Tennessee Titans': 'TEN', 'Washington Commanders': 'WAS'}
TEAM_ALIASES = {normalize_label(k):v for k,v in TEAM_NAMES.items()}
TEAM_ALIASES.update({normalize_label(v):v for v in TEAM_NAMES.values()})

def match_event(event, ranked):
    home = TEAM_ALIASES.get(normalize_label(event.get("home_team", "")))
    away = TEAM_ALIASES.get(normalize_label(event.get("away_team", "")))
    if not home or not away:
        return None
    matched = []
    for gid, group in ranked.groupby("game_id"):
        row = group.iloc[0]
        nfl_home, nfl_away = (row.team, row.opponent) if row.home else (row.opponent, row.team)
        if (home, away) != (nfl_home, nfl_away):
            continue
        # Team pair AND start time must agree; a different day's game cannot attach to this slate.
        if abs((explicit_utc(event["commence_time"]) - explicit_utc(row.start_utc)).total_seconds()) <= 3600:
            matched.append(str(gid))
    return matched[0] if len(matched) == 1 else None

def parse_event_odds(event, game_id, ranked, now=None):
    now = pd.Timestamp.now(tz="UTC") if now is None else explicit_utc(now)
    sportsbooks, underdog, issues = [], [], []
    candidates = ranked[ranked.game_id == game_id].copy()
    candidates["normalized_name"] = candidates.player.map(normalize_label)
    if match_event(event, candidates) != game_id:
        return [], [], [{"event":event.get("id"), "reason":"Event identity changed; skipped"}]
    for book in event.get("bookmakers", []):
        bkey = book.get("key")
        if bkey not in ODDS_BOOKMAKERS:
            continue
        for market in book.get("markets", []):
            if market.get("key") not in ODDS_MARKETS:
                continue
            try:
                updated = explicit_utc(market.get("last_update"))
                age = (now - updated).total_seconds() / 3600
                if age < -5/60:
                    raise ValueError("Future market timestamp")
            except (ValueError, TypeError) as err:
                issues.append({"book":bkey, "reason":str(err)}); continue
            outcomes = market.get("outcomes", [])
            for outcome in outcomes:
                try:
                    side = str(outcome.get("name", "")).casefold()
                    if side == "yes":
                        pass
                    elif side in {"over", "higher"} and float(outcome.get("point", -1)) == .5:
                        pass
                    else:
                        continue
                    source_name = str(outcome.get("description", ""))
                    model_name = PLAYER_NAME_ALIASES.get(source_name, source_name)
                    matches = candidates[candidates.normalized_name == normalize_label(model_name)]
                    if len(matches) != 1:
                        raise ValueError("Player name missing/ambiguous in this game's rankings")
                    player = matches.iloc[0]
                    if explicit_utc(player.start_utc) <= now:
                        continue
                    row = {"game_id":game_id, "player_id":str(player.player_id), "player":player.player,
                           "team":player.team, "book":book.get("title", bkey), "line":.5,
                           "quoted_at_utc":updated.isoformat(), "playing_confirmed":False,
                           "source_event_id":event.get("id"), "source_market":market["key"]}
                    if bkey == "underdog":
                        row.update(model_chance_pct=100*float(player.probability),
                                   reported_multiplier=outcome.get("multiplier", np.nan),
                                   availability="Provider-listed selection; playing status unconfirmed",
                                   prior_games=int(player.prior_games),
                                   review_eligible=confirmed_bool(player.get("review_eligible", False)),
                                   flags=str(player.get("flags", "")),
                                   note="Indicative DFS feed. Actual entry payout required for Underdog EV.")
                        underdog.append(row)   # No indicative price passed to compare_odds.
                    else:
                        american_to_decimal(outcome["price"])  # Validate before accepting.
                        row["over_american"] = outcome["price"]
                        pairs = [o for o in outcomes if str(o.get("name", "")).casefold() in ({"no"} if side == "yes" else {"under", "lower"})
                                 and o.get("description")==outcome.get("description") and o.get("point")==outcome.get("point")]
                        row["under_american"] = pairs[0].get("price", np.nan) if len(pairs)==1 else np.nan
                        sportsbooks.append(row)
                except (ValueError, TypeError, KeyError) as err:
                    issues.append({"book":bkey,"player":outcome.get("description", ""),"reason":str(err)})
    return sportsbooks, underdog, issues

def fetch_slate_odds(ranked, api_key):
    ranked = ranked[pd.to_datetime(ranked.start_utc, utc=True) > pd.Timestamp.now(tz="UTC")].copy()
    if ranked.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    events = odds_request("sports/americanfootball_nfl/events", {"dateFormat":"iso"}, api_key)
    if not isinstance(events, list):
        raise OddsFeedError("Unexpected events response.")
    mapping, issues = {}, []
    for event in events:
        try:
            gid = match_event(event, ranked)
        except (ValueError, KeyError, TypeError):
            continue
        if gid is not None:
            mapping.setdefault(gid, []).append(event)
    sportsbook_rows, underdog_rows = [], []
    for game_id in ranked.game_id.unique():
        possible = mapping.get(str(game_id), [])
        if len(possible) != 1:
            issues.append({"game_id":str(game_id),"reason":"No unique odds-provider event matched this NFL game"})
            continue
        event = possible[0]
        try:
            event_id = str(event["id"])
            if not re.fullmatch(r"[A-Za-z0-9_-]+", event_id):
                raise OddsFeedError("Invalid provider event ID.")
            response = odds_request(f"sports/americanfootball_nfl/events/{event_id}/odds",
                                     {"bookmakers":",".join(ODDS_BOOKMAKERS), "markets":",".join(ODDS_MARKETS),
                                      "oddsFormat":"american", "dateFormat":"iso", "includeMultipliers":"true"}, api_key)
            a,b,c = parse_event_odds(response, str(game_id), ranked)
            sportsbook_rows.extend(a);underdog_rows.extend(b);issues.extend(c)
        except OddsFeedError as err:
            # Key/quota/network failure stops this fetch rather than repeatedly spending requests.
            print("Automatic odds stopped:", str(err));break
    matched_ids = set(mapping)
    print(f"Odds events matched: {len(matched_ids)}/{ranked.game_id.nunique()}; "
          f"sportsbook quotes: {len(sportsbook_rows)}; Underdog selections: {len(underdog_rows)}")
    return pd.DataFrame(sportsbook_rows), pd.DataFrame(underdog_rows), pd.DataFrame(issues)



In [ ]:
# Fetch odds only for games already selected by the model.
auto_quotes, underdog_candidates, odds_feed_issues = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
value_board, rejected_odds = pd.DataFrame(), pd.DataFrame()
odds_rankings = rankings.copy()
# Attach schedule identity and UTC start time without changing saved prediction columns.
games_for_odds = selected[["game_id","home_team","kickoff"]].copy()
odds_rankings = odds_rankings.merge(games_for_odds, on="game_id", how="left", validate="many_to_one")
odds_rankings["home"] = odds_rankings.team.eq(odds_rankings.home_team)
odds_rankings["start_utc"] = pd.to_datetime(odds_rankings.kickoff, utc=True).map(lambda x:x.isoformat())
odds_rankings["probability"] = odds_rankings.any_td_probability
odds_rankings["generated_at_utc"] = pd.to_datetime(odds_rankings.generated_at_et, utc=True).map(lambda x:x.isoformat())
odds_rankings["model_fingerprint"] = fingerprint
if FETCH_ODDS and len(odds_rankings):
    _key = odds_api_key()
    if not _key:
        print("Add ODDS_API_KEY to Colab Secrets and enable notebook access; then rerun this cell.")
    else:
        try:
            _odds_client = BudgetOddsClient(BASE, _key, ODDS_REFRESH_TOKEN, ODDS_CREDIT_RESERVE, ODDS_MAX_CREDITS_PER_RUN)
            _odds_client.check_quota()
            auto_quotes, underdog_candidates, odds_feed_issues = fetch_slate_odds(odds_rankings, _key)
            print(f"New paid fetch attempts this run: {_odds_client.spent}")
        except OddsFeedError as err:
            print(str(err))
        finally:
            del _key
else:
    print("Odds fetching is off, or there are no rankings.")
if len(odds_feed_issues):
    display(odds_feed_issues)


In [ ]:
if len(auto_quotes):
    saved_prices = auto_quotes.copy()
    saved_prices["quote_age_minutes"] = (pd.Timestamp.now(tz="UTC") - pd.to_datetime(saved_prices.quoted_at_utc, utc=True)).dt.total_seconds() / 60
    saved_prices["price_status"] = np.where(saved_prices.quote_age_minutes <= MAX_ODDS_AGE_HOURS*60, "Recent quote — verify before use", "SAVED / STALE — refresh before use")
    print("SAVED PRICE SNAPSHOT — may differ from current offers:")
    display(saved_prices[["player","book","over_american","quoted_at_utc","quote_age_minutes","price_status"]].head(TOP_N))

if len(auto_quotes):
    value_board, rejected_odds = compare_odds(odds_rankings, auto_quotes)
    if len(rejected_odds):
        print("Excluded from current value board (saved prices are shown above):")
        display(rejected_odds)
    if len(value_board):
        best = value_board.drop_duplicates(["game_id","player_id"])
        cols = ["player","team","book","over_american","model_chance_pct","break_even_pct","estimated_ev_pct","buffered_ev_pct","quote_age_minutes","flags"]
        print("RAW PRICE COMPARISON — includes ineligible/low-opportunity players; not a betting shortlist. BEST SAVED PRICE PER PLAYER — rush/receiving probability applied to anytime TD odds; verify rules:")
        display(best[cols].head(TOP_N).round(2))
        print("REVIEW CANDIDATES — estimated EV >= 3%, buffered EV >= 5%, and availability/role checks:")
        display(best.loc[best.passes_value_filters, cols].head(TOP_N).round(2))
        if not best.passes_value_filters.any():
            print("No players pass the value filters. No need to force a pick.")
else:
    print("No matched anytime-TD sportsbook quotes. Check coverage and posting times; no extra market was fetched.")
if len(underdog_candidates):
    underdog_candidates["quote_age_minutes"] = (pd.Timestamp.now(tz="UTC")-pd.to_datetime(underdog_candidates.quoted_at_utc,utc=True)).dt.total_seconds()/60
    underdog_candidates["price_status"] = np.where(underdog_candidates.quote_age_minutes <= MAX_ODDS_AGE_HOURS*60,"Recent — verify in app","SAVED / STALE — refresh")
    print("UNDERDOG CANDIDATES — not single-bet EV; actual entry payout and TD rules required:")
    display(underdog_candidates[["player","team","model_chance_pct","reported_multiplier","quote_age_minutes","price_status","review_eligible","flags"]].head(TOP_N))
# Immutable snapshots, separate from training/model caches.
odds_output = BASE / "odds"
odds_output.mkdir(exist_ok=True)
stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
for label, frame in [("Quotes",auto_quotes),("Value",value_board),("Excluded",rejected_odds),("Underdog",underdog_candidates),("Feed_Issues",odds_feed_issues)]:
    if len(frame):
        frame.to_csv(odds_output / f"{stem}_{label}_{stamp}.csv", index=False)
print("Odds snapshots folder:", odds_output)

# Concise candidate list; no forced pick or parlay when the list is empty.
if len(value_board):
    card=value_board.loc[value_board.passes_value_filters].sort_values('buffered_ev_pct',ascending=False).drop_duplicates(['game_id','player_id'])
    print('SUNDAY REVIEW CARD — singles to review, not guaranteed edges:')
    if len(card):
        display(card[['player','team','book','over_american','model_chance_pct','buffered_ev_pct','flags']].head(10).round(2))
    else:
        print('No selections pass all checks at the available fresh quotes.')
    card.to_csv(OUTPUT/f'{stem}_Review_Card.csv',index=False)


## Automatic prediction history and results
Snapshots persist in Drive. Future Run all sessions grade completed games. Repeated forecasts count once per model family/player/game, using the earliest saved pregame prediction. No bet entry is needed.

In [ ]:
## Automatic forecast tracking: runs only when you run the notebook.
# No manual bet entry, no betting account access, and no odds credits for grading.
TRACKING_VERSION='forecast-tracking-v1'
tracking_dir=BASE/'evaluation'
snapshot_dir=tracking_dir/'prediction_snapshots'
snapshot_dir.mkdir(parents=True,exist_ok=True)

def atomic_csv(frame,path):
    temporary=path.with_suffix('.tmp.csv')
    frame.to_csv(temporary,index=False)
    temporary.replace(path)

def snapshot_forecasts(frame, games, folder, now):
    columns=['player_id','player','team','game_id','any_td_probability','generated_at_et','model_label']
    require(frame,columns,'Forecast snapshot')
    variants=[frame.copy()]
    if 'baseline_any_td_probability' in frame:
        baseline=frame.loc[frame.model_label.ne('original_baseline_v1')].copy()
        baseline['any_td_probability']=baseline.baseline_any_td_probability
        baseline['model_label']='original_baseline_v1'
        baseline['review_eligible']=False
        variants.append(baseline)
    if 'experimental_injury_probability' in frame:
        experimental=frame.loc[frame.experimental_injury_probability.notna() & frame.model_label.ne('injury_context_v1')].copy()
        experimental['any_td_probability']=experimental.experimental_injury_probability
        experimental['model_label']='experimental_injury_context_v1'
        experimental['review_eligible']=False
        variants.append(experimental)
    forecast=pd.concat(variants,ignore_index=True).merge(games[['game_id','kickoff']],on='game_id',how='left',validate='many_to_one')
    forecast['prediction_time_utc']=pd.to_datetime(forecast.generated_at_et,utc=True,errors='coerce')
    forecast['kickoff_utc']=pd.to_datetime(forecast.kickoff,utc=True,errors='coerce')
    now=pd.Timestamp(now).tz_convert('UTC')
    good=(forecast.prediction_time_utc.notna() & forecast.kickoff_utc.notna() &
          forecast.prediction_time_utc.lt(forecast.kickoff_utc) & forecast.prediction_time_utc.le(now) &
          forecast.kickoff_utc.gt(now) & forecast.any_td_probability.between(0,1))
    forecast=forecast.loc[good].copy()
    if forecast.empty:return None
    forecast['tracking_version']=TRACKING_VERSION
    forecast['snapshot_recorded_at_utc']=now.isoformat()
    forecast=forecast.drop(columns=['kickoff'])
    encoded=forecast[['player_id','game_id','any_td_probability','generated_at_et','model_label']].to_json().encode()
    digest=hashlib.sha256(encoded).hexdigest()[:12]
    path=folder/f'{now.strftime("%Y%m%dT%H%M%S%fZ")}_{digest}.csv'
    # Never overwrite an earlier forecast.
    if not path.exists():atomic_csv(forecast,path)
    return path

def select_first_forecasts(frames):
    if not frames:return pd.DataFrame()
    combined=pd.concat(frames,ignore_index=True)
    require(combined,['player_id','game_id','model_label','prediction_time_utc','kickoff_utc','any_td_probability'],'Saved forecasts')
    combined['prediction_time_utc']=pd.to_datetime(combined.prediction_time_utc,utc=True,errors='coerce')
    combined['kickoff_utc']=pd.to_datetime(combined.kickoff_utc,utc=True,errors='coerce')
    combined=combined.loc[combined.prediction_time_utc.lt(combined.kickoff_utc) & combined.any_td_probability.between(0,1)]
    # One forecast per model family/player/game; reruns cannot weight easy games repeatedly.
    return combined.sort_values('prediction_time_utc').drop_duplicates(['model_label','game_id','player_id'],keep='first').reset_index(drop=True)

def grade_forecasts(forecasts,games,stats,snaps,identities,now):
    result=forecasts.copy()
    result['grade']='PENDING'
    result['actual_td']=np.nan
    result['grading_note']='Waiting for completed game and complete source coverage'
    result['graded_at_utc']=pd.Timestamp(now).tz_convert('UTC').isoformat()
    completed=games.loc[games.home_score.notna() & games.away_score.notna() &
        (pd.to_datetime(games.kickoff,utc=True) < pd.Timestamp(now).tz_convert('UTC')-pd.Timedelta(hours=8))]
    ids=identities[['gsis_id','pfr_id']].dropna().drop_duplicates()
    unique=ids.groupby('gsis_id').pfr_id.nunique()
    ids=ids.loc[ids.gsis_id.isin(unique.index[unique.eq(1)])].drop_duplicates('gsis_id')
    pfr_lookup=ids.set_index('gsis_id').pfr_id.to_dict()
    for idx,row in result.iterrows():
        game=completed.loc[completed.game_id.eq(row.game_id)]
        if game.empty:continue
        expected_teams={game.iloc[0].home_team,game.iloc[0].away_team}
        game_stats=stats.loc[stats.game_id.eq(row.game_id)]
        game_snaps=snaps.loc[snaps.game_id.eq(row.game_id)]
        if not expected_teams.issubset(set(game_stats.team)) or not expected_teams.issubset(set(game_snaps.team)):
            result.at[idx,'grade']='AWAITING_DATA';continue
        scoring=game_stats.loc[game_stats.player_id.eq(row.player_id)&game_stats.team.eq(row.team)]
        if len(scoring)>1:
            result.at[idx,'grade']='DATA_REVIEW';continue
        pfr=pfr_lookup.get(row.player_id)
        if not pfr:
            result.at[idx,'grade']='IDENTITY_UNRESOLVED';continue
        participation=game_snaps.loc[game_snaps.pfr_player_id.eq(pfr)&game_snaps.team.eq(row.team)]
        played=not participation.empty and participation.offense_snaps.fillna(0).sum()>0
        if not played:
            # Missing offensive participation is excluded from conditional-model accuracy.
            # This is NOT a sportsbook settlement/refund assumption.
            if len(scoring) and (scoring.rushing_tds.fillna(0)+scoring.receiving_tds.fillna(0)).sum()>0:
                result.at[idx,'grade']='DATA_REVIEW';continue
            result.at[idx,'grade']='NO_OFFENSIVE_SNAP'
            result.at[idx,'grading_note']='Excluded from conditional model metrics; not a wager settlement'
            continue
        if scoring.empty:
            # Zero-touch players may be absent from stats. Positive snaps plus both-team data
            # supports a zero outcome; source corrections are applied again on later runs.
            actual=0
        else:
            if scoring[['rushing_tds','receiving_tds']].isna().any().any():
                result.at[idx,'grade']='AWAITING_DATA';continue
            actual=int((scoring.rushing_tds+scoring.receiving_tds).sum()>0)
        result.at[idx,'actual_td']=actual
        result.at[idx,'grade']='HIT' if actual else 'MISS'
        result.at[idx,'grading_note']='Rushing/receiving TD; offensive participant'
    return result

saved_snapshot=snapshot_forecasts(rankings,selected,snapshot_dir,pd.Timestamp.now(tz='UTC'))
print('Forecast snapshot saved:',saved_snapshot is not None)
frames=[]
for path in sorted(snapshot_dir.glob('*.csv')):
    if path.name.endswith('.tmp.csv'):continue
    try:frames.append(pd.read_csv(path))
    except Exception as err:print('Unreadable forecast snapshot:',path.name,type(err).__name__)
tracked=select_first_forecasts(frames)
if len(tracked):
    stats_parts=[];snaps_parts=[]
    # NFL games in January belong to the season in game_id, not their calendar year.
    years=sorted(set(tracked.game_id.astype(str).str.split('_').str[0].astype(int)))
    for year in years:
        try:
            # Separate result data: never inserted into this week's training features.
            raw=nfl.load_player_stats(int(year))
            st=raw.to_pandas() if hasattr(raw,'to_pandas') else raw
            raw=nfl.load_snap_counts(int(year))
            sn=raw.to_pandas() if hasattr(raw,'to_pandas') else raw
            require(st,['game_id','team','player_id','rushing_tds','receiving_tds'],'Grading stats')
            require(sn,['game_id','team','pfr_player_id','offense_snaps'],'Grading snap counts')
            stats_parts.append(st);snaps_parts.append(sn)
        except Exception as err:
            print(f'Results not ready for {year}: {type(err).__name__}. Those forecasts stay pending.')
    st=pd.concat(stats_parts,ignore_index=True) if stats_parts else pd.DataFrame(columns=['game_id','team','player_id','rushing_tds','receiving_tds'])
    sn=pd.concat(snaps_parts,ignore_index=True) if snaps_parts else pd.DataFrame(columns=['game_id','team','pfr_player_id','offense_snaps'])
    identities=pd.concat([player_directory[['gsis_id','pfr_id']],current_rosters[['gsis_id','pfr_id']]],ignore_index=True)
    graded=grade_forecasts(tracked,schedule,st,sn,identities,pd.Timestamp.now(tz='UTC'))
    atomic_csv(graded,tracking_dir/'Graded_Predictions.csv')
    print('Forecast grading:',graded.grade.value_counts().to_dict())
    scored=graded.loc[graded.grade.isin(['HIT','MISS'])].copy()
    if len(scored):
        scored['squared_error']=(scored.any_td_probability-scored.actual_td)**2
        scored['probability_band']=pd.cut(scored.any_td_probability,[0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1],include_lowest=True)
        summary=scored.groupby('model_label').agg(predictions=('actual_td','size'),games=('game_id','nunique'),predicted_rate=('any_td_probability','mean'),actual_rate=('actual_td','mean'),Brier=('squared_error','mean')).reset_index()
        bands=scored.groupby(['model_label','probability_band'],observed=True).agg(predictions=('actual_td','size'),predicted_rate=('any_td_probability','mean'),actual_rate=('actual_td','mean'),Brier=('squared_error','mean')).reset_index()
        atomic_csv(summary,tracking_dir/'Performance_Summary.csv')
        atomic_csv(bands,tracking_dir/'Calibration_By_Probability.csv')
        display(summary.round(4));display(bands.round(4))
    print('Results update on a later Run all. No personal profit or simulated historical bets are inferred.')
